# Data Extraction Module for Pricing & Offers System


## Imports


In [1]:
!pip install slack

In [2]:
import os
import numpy as np
import pandas as pd
import snowflake.connector
import setup_environment_2
import slack
import pytz
from common_functions import upload_dataframe_to_snowflake,send_text_slack

# Cairo timezone for consistent timestamps
CAIRO_TZ = pytz.timezone('Africa/Cairo')


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(
/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/slack/deprecation.py:14: UserWarning: slack package is deprecated. Please use slack_sdk.web/webhook/rtm package instead. For more info, go to https://docs.slack.dev/tools/python-slack-sdk/v3-migration/
  warnings.warn(message)


## Variables


In [3]:
# Region
REGION = "Egypt"

# Snowflake Warehouse
WAREHOUSE = "COMPUTE_WH"

# Date Variables
from datetime import datetime, timedelta
TODAY = datetime.now(CAIRO_TZ).date()
YESTERDAY = TODAY - timedelta(days=1)


## Initialize Environment


In [4]:
setup_environment_2.initialize_env()


/home/ec2-user/.Renviron


/home/ec2-user/service_account_key.json


## Warehouse & Cohort Mapping


In [5]:
# Warehouse Mapping: (region, warehouse_name, warehouse_id, cohort_id)
WAREHOUSE_MAPPING = [
    ('Cairo', 'Mostorod', 1, 700),
    ('Giza', 'Barageel', 236, 701),
    ('Giza', 'Sakkarah', 962, 701),
    ('Delta West', 'El-Mahala', 337, 703),
    ('Delta West', 'Tanta', 8, 703),
    ('Delta East', 'Mansoura FC', 339, 704),
    ('Delta East', 'Sharqya', 170, 704),
    ('Upper Egypt', 'Assiut FC', 501, 1124),
    ('Upper Egypt', 'Bani sweif', 401, 1126),
    ('Upper Egypt', 'Menya Samalot', 703, 1123),
    ('Upper Egypt', 'Sohag', 632, 1125),
    ('Alexandria', 'Khorshed Alex', 797, 702),
]



# All Cohort IDs
COHORT_IDS = [700, 701, 702, 703, 704, 1123, 1124, 1125, 1126]


## Snowflake Query Function


In [6]:
def query_snowflake(query):
    """Execute a query on Snowflake and return results as DataFrame."""
    con = snowflake.connector.connect(
        user=os.environ["SNOWFLAKE_USERNAME"],
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        password=os.environ["SNOWFLAKE_PASSWORD"],
        database=os.environ["SNOWFLAKE_DATABASE"]
    )
    try:
        cur = con.cursor()
        cur.execute("USE WAREHOUSE COMPUTE_WH")
        cur.execute(query)
        data = cur.fetchall()
        columns = [desc[0].lower() for desc in cur.description]  # Get column names from cursor
        return pd.DataFrame(data, columns=columns)
    except Exception as e:
        print(f"Snowflake Error: {e}")
        return pd.DataFrame()
    finally:
        cur.close()
        con.close()


In [7]:
def get_snowflake_timezone():
    """Get the current timezone from Snowflake."""
    query = "SHOW PARAMETERS LIKE 'TIMEZONE'"
    result = query_snowflake(query)
    return result.value[0] if len(result) > 0 else "UTC"


## Helper Functions


In [8]:
def get_warehouse_df():
    """Get warehouse mapping as DataFrame."""
    return pd.DataFrame(
        WAREHOUSE_MAPPING,
        columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
    


def convert_to_numeric(df):
    """Convert DataFrame columns to numeric where possible."""
    df.columns = df.columns.str.lower()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')
    return df


## Get Snowflake Timezone


In [9]:
TIMEZONE = get_snowflake_timezone()
print(f"Snowflake timezone: {TIMEZONE}")


Snowflake timezone: America/Los_Angeles


In [10]:
# =============================================================================
# 8. ALL-TIME HIGH MARGIN QUERY (P80 margin weighted by gross profit)
# This calculates the top 80% margin based on days with best gross profit
# Gross Profit = NMV × margin, so high-margin + high-sales days rank higher
# =============================================================================
ALL_TIME_HIGH_MARGIN_QUERY = f'''
WITH daily_margin_data AS (
    SELECT 
        pso.warehouse_id,
        pso.product_id,
        so.created_at::DATE AS sale_date,
        SUM(pso.total_price) AS daily_nmv,
        SUM(COALESCE(f.wac_p, 0) * pso.purchased_item_count * pso.basic_unit_count) AS daily_cogs,
        CASE 
            WHEN SUM(pso.total_price) > 0 
            THEN (SUM(pso.total_price) - SUM(COALESCE(f.wac_p, 0) * pso.purchased_item_count * pso.basic_unit_count)) / SUM(pso.total_price)
            ELSE 0 
        END AS daily_margin
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    JOIN finance.all_cogs f ON f.product_id = pso.product_id
        AND f.from_date::DATE <= so.created_at::DATE 
        AND f.to_date::DATE > so.created_at::DATE
    WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 240
        AND so.created_at::DATE < CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY pso.warehouse_id, pso.product_id, so.created_at::DATE
),

-- Calculate gross profit and rank days by it
ranked_by_gross_profit AS (
    SELECT 
        warehouse_id,
        product_id,
        sale_date,
        daily_nmv,
        daily_margin,
        daily_nmv * daily_margin AS gross_profit,
        -- Rank by gross profit (1 = highest gross profit day)
        ROW_NUMBER() OVER (
            PARTITION BY warehouse_id, product_id 
            ORDER BY daily_nmv * daily_margin DESC
        ) AS gp_rank,
        COUNT(*) OVER (PARTITION BY warehouse_id, product_id) AS total_days
    FROM daily_margin_data
    WHERE daily_nmv > 0 AND daily_margin > 0
)

-- Take P80 of margins from TOP-ranked days by gross profit
-- P80 = take the 80th percentile of margins from the best-performing days
SELECT 
    warehouse_id,
    product_id,
    -- P80 margin from days ranked by gross profit
    PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY daily_margin) AS all_time_high_margin,
    -- Also provide some context stats
    MAX(daily_margin) AS max_daily_margin,
    AVG(daily_margin) AS avg_daily_margin,
    COUNT(*) AS days_with_profit
FROM ranked_by_gross_profit
-- Include only top 80% of days by gross profit rank
WHERE gp_rank <= GREATEST(1, CEIL(total_days * 0.8))
GROUP BY warehouse_id, product_id
'''

print("All-time high margin query defined (P80 margin weighted by gross profit)")


All-time high margin query defined (P80 margin weighted by gross profit)


## Market Prices Extraction Queries
Queries for external market price data:
1. **Ben Soliman Prices** - Competitor reference prices
2. **Marketplace Prices** - Min, Max, Mod prices from marketplace
3. **Scrapped Data** - Competitor prices from scraping


In [11]:
## Additional Data Queries (Sales, Groups, WAC)


In [12]:
# =============================================================================
# 4. PRODUCT BASE DATA QUERY (product_id, sku, brand, cat, wac1, wac_p, current_price)
# =============================================================================
PRODUCT_BASE_QUERY = f'''
WITH skus_prices AS (
    WITH local_prices AS (
        SELECT  
            CASE 
                WHEN cpu.cohort_id IN (700, 695) THEN 'Cairo'
                WHEN cpu.cohort_id IN (701) THEN 'Giza'
                WHEN cpu.cohort_id IN (704, 698) THEN 'Delta East'
                WHEN cpu.cohort_id IN (703, 697) THEN 'Delta West'
                WHEN cpu.cohort_id IN (696, 1123, 1124, 1125, 1126) THEN 'Upper Egypt'
                WHEN cpu.cohort_id IN (702, 699) THEN 'Alexandria'
            END AS region,
            cohort_id,
            pu.product_id,
            pu.packing_unit_id,
            pu.basic_unit_count,
            AVG(cpu.price) AS price
        FROM cohort_product_packing_units cpu
        JOIN PACKING_UNIT_PRODUCTS pu ON pu.id = cpu.product_packing_unit_id
        WHERE cpu.cohort_id IN (700,701,702,703,704,695,696,697,698,699,1123,1124,1125,1126)
            AND cpu.created_at::date <> '2023-07-31'
            AND cpu.is_customized = TRUE
        GROUP BY ALL
    ),
    
    live_prices AS (
        SELECT 
            region, cohort_id, product_id, 
            pu_id AS packing_unit_id, 
            buc AS basic_unit_count, 
            NEW_PRICE AS price
        FROM materialized_views.DBDP_PRICES
        WHERE created_at = CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date
            AND DATE_PART('hour', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::time) 
                BETWEEN SPLIT_PART(time_slot, '-', 1)::int AND (SPLIT_PART(time_slot, '-', 1)::int) + 1
            AND cohort_id IN (700,701,702,703,704,695,696,697,698,699,1123,1124,1125,1126)
    ),
    
    prices AS (
        SELECT *
        FROM (
            SELECT *, 1 AS priority FROM live_prices
            UNION ALL
            SELECT *, 2 AS priority FROM local_prices
        )
        QUALIFY ROW_NUMBER() OVER (PARTITION BY region, cohort_id, product_id, packing_unit_id ORDER BY priority) = 1
    )
    
    SELECT region, cohort_id, product_id, price
    FROM prices
    WHERE basic_unit_count = 1
        AND ((product_id = 1309 AND packing_unit_id = 2) OR (product_id <> 1309))
)

SELECT distinct
    region, cohort_id, p.product_id,
    CONCAT(products.name_ar, ' ', products.size, ' ', product_units.name_ar) AS sku,
    b.name_ar AS brand,
    cat.name_ar AS cat,
    wac1, wac_p, p.price as current_price
FROM skus_prices p
JOIN finance.all_cogs c ON c.product_id = p.product_id 
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN c.from_date AND c.to_date
JOIN products ON products.id = p.product_id
JOIN categories cat ON cat.id = products.category_id
JOIN brands b ON b.id = products.brand_id
JOIN product_units ON product_units.id = products.unit_id
WHERE wac1 > 0 AND wac_p > 0
GROUP BY ALL
'''

# =============================================================================
# 5. SALES DATA QUERY (120-day NMV by cohort/product)
# =============================================================================
SALES_QUERY = f'''
SELECT DISTINCT cpc.cohort_id, pso.product_id,
    CONCAT(products.name_ar,' ',products.size,' ',product_units.name_ar) as sku,
    brands.name_ar as brand, categories.name_ar as cat,
    sum(pso.total_price) as nmv
FROM product_sales_order pso
JOIN sales_orders so ON so.id = pso.sales_order_id
JOIN COHORT_PRICING_CHANGES cpc ON cpc.id = pso.COHORT_PRICING_CHANGE_id
JOIN products ON products.id = pso.product_id
JOIN brands ON products.brand_id = brands.id 
JOIN categories ON products.category_id = categories.id
JOIN product_units ON product_units.id = products.unit_id 
WHERE so.created_at::date BETWEEN CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 120 
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 1 
    AND so.sales_order_status_id NOT IN (7, 12)
    AND so.channel IN ('telesales', 'retailer')
    AND pso.purchased_item_count <> 0
    AND cpc.cohort_id IN (700,701,702,703,704,1123,1124,1125,1126)
GROUP BY ALL
'''

# =============================================================================
# 6. MARGIN STATS QUERY (STD and average margins)  
# =============================================================================
MARGIN_STATS_QUERY = f'''
select product_id, cohort_id, 
    (0.6*product_std) + (0.3*brand_std) + (0.1*cat_std) as std, 
    avg_margin
from (
    select product_id, cohort_id, 
        stddev(product_margin) as product_std, 
        stddev(brand_margin) as brand_std,
        stddev(cat_margin) as cat_std, 
        avg(product_margin) as avg_margin
    from (
        select distinct product_id, order_date, cohort_id,
            (nmv-cogs_p)/nmv as product_margin, 
            (brand_nmv-brand_cogs)/brand_nmv as brand_margin,
            (cat_nmv-cat_cogs)/cat_nmv as cat_margin
        from (
            SELECT DISTINCT so.created_at::date as order_date, cpc.cohort_id, pso.product_id,
                brands.name_ar as brand, categories.name_ar as cat,
                sum(COALESCE(f.wac_p,0) * pso.purchased_item_count * pso.basic_unit_count) as cogs_p,
                sum(pso.total_price) as nmv,
                sum(nmv) over(partition by order_date, cat, brand) as brand_nmv,
                sum(cogs_p) over(partition by order_date, cat, brand) as brand_cogs,
                sum(nmv) over(partition by order_date, cat) as cat_nmv,
                sum(cogs_p) over(partition by order_date, cat) as cat_cogs
            FROM product_sales_order pso
            JOIN sales_orders so ON so.id = pso.sales_order_id   
            JOIN COHORT_PRICING_CHANGES cpc on cpc.id = pso.cohort_pricing_change_id
            JOIN products on products.id = pso.product_id
            JOIN brands on products.brand_id = brands.id 
            JOIN categories ON products.category_id = categories.id
            JOIN finance.all_cogs f ON f.product_id = pso.product_id
                AND f.from_date::date <= so.created_at::date AND f.to_date::date > so.created_at::date
            WHERE so.created_at::date between 
                date_trunc('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 120) 
                and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date
                AND so.sales_order_status_id not in (7,12)
                AND so.channel IN ('telesales','retailer')
                AND pso.purchased_item_count <> 0
            GROUP BY ALL
        )
    ) group by all 
)
'''

# =============================================================================
# 7. TARGET MARGINS QUERY
# =============================================================================
TARGET_MARGINS_QUERY = f'''
WITH cat_brand_target as (
    SELECT DISTINCT cat, brand, margin as target_bm
    FROM performance.commercial_targets cplan
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(DATE) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
),
cat_target as (
    select cat, sum(target_bm * (target_nmv/cat_total)) as cat_target_margin
    from (
        select *, sum(target_nmv) over(partition by cat) as cat_total
        from (
            select cat, brand, avg(target_bm) as target_bm, sum(target_nmv) as target_nmv
            from (
                SELECT DISTINCT date, city as region, cat, brand, margin as target_bm, nmv as target_nmv
                FROM performance.commercial_targets cplan
                QUALIFY CASE 
                    WHEN DATE_TRUNC('month', MAX(DATE) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date) 
                    THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date)
                    ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - INTERVAL '1 month') 
                END = DATE_TRUNC('month', date)
            ) group by all
        )
    ) group by all 
)
SELECT DISTINCT cbt.cat, cbt.brand, cbt.target_bm, ct.cat_target_margin
FROM cat_brand_target cbt
LEFT JOIN cat_target ct ON ct.cat = cbt.cat
'''


In [13]:
## Execute All Queries


In [14]:
# =============================================================================
# Execute all queries
# =============================================================================
print("Loading data from Snowflake...")

# NOTE: Ben Soliman, Marketplace, and Scrapped prices are now fetched via
# market_data_module.ipynb get_market_data() function - no need to load here

# 1. Product Base Data (product_id, sku, brand, cat, wac1, wac_p, current_price)
print("  1. Loading product base data...")
df_product_base = query_snowflake(PRODUCT_BASE_QUERY)
df_product_base = convert_to_numeric(df_product_base)
print(f"     Loaded {len(df_product_base)} product base records")

# 2. Sales Data
print("  2. Loading sales data...")
df_sales = query_snowflake(SALES_QUERY)
df_sales = convert_to_numeric(df_sales)
print(f"     Loaded {len(df_sales)} sales records")

# 3. Margin Stats
print("  3. Loading margin stats...")
df_margin_stats = query_snowflake(MARGIN_STATS_QUERY)
df_margin_stats = convert_to_numeric(df_margin_stats)
print(f"     Loaded {len(df_margin_stats)} margin stat records")

# 4. Target Margins
print("  4. Loading target margins...")
df_targets = query_snowflake(TARGET_MARGINS_QUERY)
df_targets = convert_to_numeric(df_targets)
print(f"     Loaded {len(df_targets)} target margin records")

# 5. Product Groups (from PostgreSQL)
print("  5. Loading product groups...")
df_groups = query_snowflake(
    '''SELECT * FROM materialized_views.sku_commercial_groups'''
)
df_groups.columns = df_groups.columns.str.lower()
df_groups = convert_to_numeric(df_groups)
print(f"     Loaded {len(df_groups)} group records")

# 6. All-Time High Margin (P80 margin weighted by gross profit)
print("  6. Loading all-time high margin data...")
df_all_time_high_margin = query_snowflake(ALL_TIME_HIGH_MARGIN_QUERY)
df_all_time_high_margin = convert_to_numeric(df_all_time_high_margin)
print(f"     Loaded {len(df_all_time_high_margin)} all-time high margin records")

print("\nBase queries completed!")
print("NOTE: Market data (Ben Soliman, Marketplace, Scrapped) will be fetched via market_data_module")
print(f"\n{'='*60}")
print("df_product_base DataFrame available with columns:")
print("  - region, cohort_id, product_id, sku, brand, cat, wac1, wac_p, current_price")
print(f"{'='*60}")


Loading data from Snowflake...
  1. Loading product base data...


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


     Loaded 102958 product base records
  2. Loading sales data...


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


     Loaded 21233 sales records
  3. Loading margin stats...


...

     Loaded 471 target margin records
  5. Loading product groups...


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


     Loaded 2937 group records
  6. Loading all-time high margin data...


     Loaded 36584 all-time high margin records

Base queries completed!
NOTE: Market data (Ben Soliman, Marketplace, Scrapped) will be fetched via market_data_module

df_product_base DataFrame available with columns:
  - region, cohort_id, product_id, sku, brand, cat, wac1, wac_p, current_price


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [15]:
# =============================================================================
# PART A: Get market_data from market_data_module
# =============================================================================
# Instead of duplicating the market data processing logic here, 
# we use the centralized market_data_module which handles:
# - Ben Soliman prices
# - Marketplace prices  
# - Scrapped prices
# - Group-level price aggregation
# - Price coverage filtering
# - Price percentile calculation
# - Margin tier conversion
# =============================================================================

print("Loading market_data from market_data_module...")
print("(This fetches Ben Soliman, Marketplace, and Scrapped prices from Snowflake)")
print()

# Run market_data_module to get access to get_market_data() function
%run modules/market_data_module.ipynb

# Get fresh market data using the module (no input required)
market_data = get_market_data()

# The market_data now contains:
# - cohort_id, product_id, region
# - Raw prices: ben_soliman_price, final_min_price, final_max_price, etc.
# - Percentiles: minimum, percentile_25, percentile_50, percentile_75, maximum
# - Margins: below_market, market_min, market_25, market_50, market_75, market_max, above_market

print(f"\n{'='*60}")
print(f"MARKET DATA LOADED FROM MODULE")
print(f"{'='*60}")
print(f"Total records: {len(market_data)}")
print(f"  - With marketplace prices: {(~market_data['final_min_price'].isna()).sum()}")
print(f"  - With scrapped prices: {(~market_data['min_scrapped'].isna()).sum()}")
print(f"  - With Ben Soliman prices: {(~market_data['ben_soliman_price'].isna()).sum()}")


Loading market_data from market_data_module...
(This fetches Ben Soliman, Marketplace, and Scrapped prices from Snowflake)



/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


Market Data Module loaded at 2026-02-26 08:01:57 Cairo time
Snowflake timezone: America/Los_Angeles
All queries defined ✓
Helper functions defined ✓
get_market_data() function defined ✓
get_margin_tiers() function defined ✓

MARKET DATA MODULE READY

Available functions (NO INPUT REQUIRED):
  - get_market_data()   : Fetch and process all market prices
  - get_margin_tiers()  : Fetch and calculate margin tiers

Usage:
  %run market_data_module.ipynb
  df_market = get_market_data()
  df_tiers = get_margin_tiers()

FETCHING MARKET DATA
Timestamp: 2026-02-26 08:01:58 Cairo time

Step 1: Fetching raw price data...
  1.1 Ben Soliman prices...


      Loaded 1553 records
  1.2 Marketplace prices...


      Loaded 10960 records
  1.3 Scrapped prices...


...

      Loaded 471 records
  1.8 Product base (WAC)...


      Loaded 66042 records

Step 2: Joining all market price sources (outer join)...
    Market prices base: 14743 records

Step 3: Adding cohort IDs and supporting data...
    Records after adding cohorts: 22108

Step 4: Processing group-level prices...


/tmp/ipykernel_25840/3245917641.py:139: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/tmp/ipykernel_25840/3245917641.py:149: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged[col] = merged[col].fillna(merged[f'{col}_group'])


    Records after group processing: 26012

Step 5: Adding WAC and margin data...
    Records with WAC: 26066

Step 6: Filtering by price coverage...
    Records after price coverage filter: 12492

Step 7: Calculating price percentiles...


    Records after price analysis: 11293

Step 8: Converting prices to margins...

MARKET DATA COMPLETE
Total records: 11293
  - With marketplace prices: 11293
  - With scrapped prices: 0
  - With Ben Soliman prices: 8130

MARKET DATA LOADED FROM MODULE
Total records: 11293
  - With marketplace prices: 11293
  - With scrapped prices: 0
  - With Ben Soliman prices: 8130


## PART A: Build Main pricing_data DataFrame
Start with df_product_base (all our SKUs) and LEFT JOIN the processed market_data


In [16]:
# =============================================================================
# PART B: Build Main pricing_data DataFrame from df_product_base
# =============================================================================
print("Building main pricing_data DataFrame...")

# =============================================================================
# Step 1: Start with df_product_base as the MAIN dataframe (all our SKUs)
# =============================================================================
print("  Step 1: Starting with product base (all SKUs)...")
pricing_data = df_product_base.copy()
print(f"     Product base: {len(pricing_data)} records")

# =============================================================================
# Step 2: Add warehouse mapping (warehouse_id and warehouse name)
# =============================================================================
print("  Step 2: Adding warehouse mapping...")
warehouse_df = get_warehouse_df()
pricing_data = pricing_data.merge(
    warehouse_df[['cohort_id', 'warehouse_id', 'warehouse']], 
    on='cohort_id'
)
print(f"     After warehouse mapping: {len(pricing_data)} records")

# =============================================================================
# Step 3: LEFT JOIN processed market_data
# =============================================================================
print("  Step 3: Left joining processed market data...")
pricing_data = pricing_data.merge(
    market_data, 
    on=['cohort_id', 'product_id','region'], 
    how='left'
)
print(f"     After market data join: {len(pricing_data)} records")

# =============================================================================
# Step 4: LEFT JOIN supporting data (sales, margins, targets, groups)
# =============================================================================
print("  Step 4: Left joining supporting data...")

# Merge sales data (nmv only)
pricing_data = pricing_data.merge(
    df_sales[['cohort_id', 'product_id', 'nmv']], 
    on=['cohort_id', 'product_id'], 
    how='left'
)
pricing_data['nmv'] = pricing_data['nmv'].fillna(0)

# Merge margin statistics (by cohort_id + product_id)
pricing_data = pricing_data.merge(df_margin_stats, on=['cohort_id', 'product_id'], how='left')

# Merge target margins (by brand + cat)
pricing_data = pricing_data.merge(df_targets, on=['brand', 'cat'], how='left')
pricing_data['target_margin'] = pricing_data['target_bm'].fillna(pricing_data['cat_target_margin']).fillna(0)
pricing_data = pricing_data.drop(columns=['target_bm', 'cat_target_margin'], errors='ignore')

# Fill NaN values with defaults
pricing_data['std'] = pricing_data['std'].fillna(0.01)
pricing_data['avg_margin'] = pricing_data['avg_margin'].fillna(0)

# Merge product groups
pricing_data = pricing_data.merge(df_groups, on='product_id', how='left')

# =============================================================================
# Step 5: Calculate current margin
# =============================================================================
pricing_data['current_margin'] = (pricing_data['current_price'] - pricing_data['wac_p']) / pricing_data['current_price']

# Remove duplicates
pricing_data = pricing_data.drop_duplicates(subset=['cohort_id', 'product_id','warehouse_id'])

# =============================================================================
# Reorder columns
# =============================================================================
final_columns = [
    # Product Base Info
    'cohort_id', 'product_id', 'region', 'warehouse_id', 'warehouse', 'sku', 'brand', 'cat',
    # Cost & Price
    'wac1', 'wac_p', 'current_price', 'current_margin',
    # Sales
    'nmv',
    # Market Prices (raw)
    'ben_soliman_price', 
    'final_min_price', 'final_max_price', 'final_mod_price', 'final_true_min', 'final_true_max',
    'min_scrapped', 'scrapped25', 'scrapped50', 'scrapped75', 'max_scrapped',
    # Price Percentiles
    'minimum', 'percentile_25', 'percentile_50', 'percentile_75', 'maximum',
    # Margin Tiers
    'below_market', 'market_min', 'market_25', 'market_50', 'market_75', 'market_max', 'above_market',
    # Supporting Data
    'std', 'avg_margin', 'target_margin', 'group'
]
pricing_data = pricing_data[[c for c in final_columns if c in pricing_data.columns]]

print(f"\n{'='*60}")
print(f"PRICING DATA COMPLETE")
print(f"{'='*60}")
print(f"Total records: {len(pricing_data)}")
print(f"\nRecords with market data: {len(pricing_data[~pricing_data['minimum'].isna()])}")
print(f"Records without market data: {len(pricing_data[pricing_data['minimum'].isna()])}")
print(f"\nRecords with sales (nmv > 0): {len(pricing_data[pricing_data['nmv'] > 0])}")
print(f"Records without sales (nmv = 0): {len(pricing_data[pricing_data['nmv'] == 0])}")
print(f"\nSample data:")
pricing_data.head()


Building main pricing_data DataFrame...
  Step 1: Starting with product base (all SKUs)...
     Product base: 102958 records
  Step 2: Adding warehouse mapping...
     After warehouse mapping: 87131 records
  Step 3: Left joining processed market data...
     After market data join: 87430 records
  Step 4: Left joining supporting data...



PRICING DATA COMPLETE
Total records: 87123

Records with market data: 14136
Records without market data: 72987

Records with sales (nmv > 0): 30030
Records without sales (nmv = 0): 57093

Sample data:


,cohort_id,product_id,region,warehouse_id,warehouse,sku,brand,cat,wac1,wac_p,...,below_market,market_min,market_25,market_50,market_75,market_max,above_market,std,avg_margin,target_margin
0,702,4940,Alexandria,797,Khorshed Alex,مناديل فاميليا مطبخ كلاسيك 6 رول + 2 رول هدية ...,فاميليا,ورقيات,252.179432,229.623383,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008899,0.052518,0.064115
1,1125,22318,Upper Egypt,632,Sohag,اوريو بسكوت بكريمة بطعم الشوكولاتة والبندق - ...,اوريو,بسكويت و معمول,99.250000,92.256051,...,0.072804,0.077439,0.078821,0.086574,0.094197,0.095529,0.099941,0.033704,0.034993,0.090570
3,702,788,Alexandria,797,Khorshed Alex,حلوانى مربى فراولة- 340 جم,حلواني,مربي,404.982797,402.791186,...,-0.009502,-0.009502,-0.004467,-0.002217,0.015421,0.017582,0.024128,0.029008,0.019531,0.030954
5,700,21683,Cairo,1,Mostorod,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,لمبادا,ويفر,46.500000,45.571274,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.016206,0.067193,0.085251
7,701,9786,Giza,236,Barageel,عصير جهينة بيور جوافة كوكتيل - 235 مل,جهينة عصاير,عصاير,404.642320,385.421810,...,0.040643,0.050685,0.060520,0.070152,0.081782,0.093125,0.102104,0.009537,0.055642,0.050000


## Discount Analysis - Price & Margin After Discount


In [17]:
# =============================================================================
# Discount Query - Get discount percentage by warehouse and product
# =============================================================================
DISCOUNT_QUERY = f'''
SELECT warehouse_id, product_id, total_discount/total_nmv AS discount_perc
FROM (
    SELECT  
        pso.warehouse_id,
        pso.product_id,
        SUM(pso.total_price) AS total_nmv,
        SUM((ITEM_QUANTITY_DISCOUNT_VALUE * pso.purchased_item_count) + 
            (ITEM_DISCOUNT_VALUE * pso.purchased_item_count)) AS total_discount
    FROM product_sales_order pso 
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY ALL
)
WHERE total_nmv > 0
'''

# Execute discount query
print("Loading discount data...")
df_discount = query_snowflake(DISCOUNT_QUERY)
df_discount = convert_to_numeric(df_discount)
print(f"Loaded {len(df_discount)} discount records")


Loading discount data...


Loaded 12366 discount records


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [18]:
# =============================================================================
# Create pricing_with_discount DataFrame
# =============================================================================
print("Creating pricing_with_discount DataFrame...")

# Copy pricing_data
pricing_with_discount = pricing_data.copy()

# Merge discount data (by warehouse_id + product_id)
pricing_with_discount = pricing_with_discount.merge(
    df_discount[['warehouse_id', 'product_id', 'discount_perc']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing discount_perc with 0 (no discount)
pricing_with_discount['discount_perc'] = pricing_with_discount['discount_perc'].fillna(0)

# Merge all-time high margin data (P80 margin weighted by gross profit)
pricing_with_discount = pricing_with_discount.merge(
    df_all_time_high_margin[['warehouse_id', 'product_id', 'all_time_high_margin']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing all_time_high_margin with target_margin (fallback)
pricing_with_discount['all_time_high_margin'] = pricing_with_discount['all_time_high_margin'].fillna(
    pricing_with_discount['target_margin']
)

# =============================================================================
# Calculate price and margin after discount
# =============================================================================
# Price after discount = current_price * (1 - discount_perc)
pricing_with_discount['price_after_discount'] = (
    pricing_with_discount['current_price'] * (1 - pricing_with_discount['discount_perc'])
)

# Margin after discount = (price_after_discount - wac_p) / price_after_discount
pricing_with_discount['margin_after_discount'] = (
    (pricing_with_discount['price_after_discount'] - pricing_with_discount['wac_p']) / 
    pricing_with_discount['price_after_discount']
)

print(f"\n{'='*60}")
print(f"PRICING WITH DISCOUNT DATA COMPLETE")
print(f"{'='*60}")
print(f"Total records: {len(pricing_with_discount)}")
print(f"Records with discount (discount_perc > 0): {len(pricing_with_discount[pricing_with_discount['discount_perc'] > 0])}")
print(f"Records without discount: {len(pricing_with_discount[pricing_with_discount['discount_perc'] == 0])}")
print(f"\nNew columns added:")
print("  - discount_perc: discount percentage from sales")
print("  - price_after_discount: current_price * (1 - discount_perc)")
print("  - margin_after_discount: (price_after_discount - wac_p) / price_after_discount")
print(f"\nSample data with discounts:")
pricing_with_discount[pricing_with_discount['discount_perc'] > 0][
    ['product_id', 'warehouse_id', 'current_price', 'current_margin', 
     'discount_perc', 'price_after_discount', 'margin_after_discount']
].head(10)


Creating pricing_with_discount DataFrame...

PRICING WITH DISCOUNT DATA COMPLETE
Total records: 87123
Records with discount (discount_perc > 0): 4906
Records without discount: 82217

New columns added:
  - discount_perc: discount percentage from sales
  - price_after_discount: current_price * (1 - discount_perc)
  - margin_after_discount: (price_after_discount - wac_p) / price_after_discount

Sample data with discounts:


,product_id,warehouse_id,current_price,current_margin,discount_perc,price_after_discount,margin_after_discount
4,9786,236,406.00,0.050685,0.003100,404.741375,0.047733
5,9786,962,406.00,0.050685,0.003055,404.759720,0.047776
8,8635,703,85.00,0.020840,0.011800,83.997000,0.009148
9,362,1,209.50,0.049328,0.000484,209.398640,0.048868
10,3889,797,255.00,0.077250,0.007274,253.145189,0.070489
18,8635,236,86.25,0.035031,0.014605,84.990346,0.020729
19,8635,962,86.25,0.035031,0.001019,86.162104,0.034047
31,151,962,438.75,0.025472,0.001457,438.110869,0.024050
33,2057,703,61.75,0.013839,0.010087,61.127112,0.003790
36,62,236,87.00,0.080662,0.007162,86.376942,0.074031


In [19]:
# =============================================================================
# Price Position - Determine where price_after_discount falls in market tiers
# =============================================================================

def get_price_position(row):
    """Determine the price position relative to market price tiers."""
    price = row['price_after_discount']
    wac = row['wac_p']
    
    # Check if we have market data (minimum price exists)
    if pd.isna(row['minimum']) or pd.isna(price):
        return "No Market Data"
    
    # Get price tiers
    min_price = row['minimum']
    p25 = row['percentile_25']
    p50 = row['percentile_50']
    p75 = row['percentile_75']
    max_price = row['maximum']
    
    # Calculate below_market and above_market prices from margins
    # margin = (price - wac) / price  =>  price = wac / (1 - margin)
    below_market_margin = row['below_market']
    above_market_margin = row['above_market']
    
    below_market_price = wac / (1 - below_market_margin) if below_market_margin < 1 else min_price
    above_market_price = wac / (1 - above_market_margin) if above_market_margin < 1 else max_price
    
    # Determine position based on price tiers
    if price < below_market_price:
        return "Below Market"
    elif price < min_price:
        return "Below Min"
    elif price < p25:
        return "At Min"
    elif price < p50:
        return "At 25th"
    elif price < p75:
        return "At 50th"
    elif price < max_price:
        return "At 75th"
    elif price < above_market_price:
        return "At Max"
    else:
        return "Above Market"

# Apply price position function
pricing_with_discount['price_position'] = pricing_with_discount.apply(get_price_position, axis=1)

# Summary of price positions
print(f"\n{'='*60}")
print(f"PRICE POSITION ANALYSIS")
print(f"{'='*60}")
print("\nPrice Position Distribution:")
print(pricing_with_discount['price_position'].value_counts().to_string())
print(f"\nPrice Position Percentages:")
print((pricing_with_discount['price_position'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

# Sample data showing price position
print(f"\nSample data with price position:")
pricing_with_discount[
    ['product_id', 'warehouse_id', 'sku', 'current_price', 'discount_perc', 
     'price_after_discount', 'minimum', 'maximum', 'price_position']
].head(15)



PRICE POSITION ANALYSIS

Price Position Distribution:
price_position
No Market Data    72987
Below Market       4975
At Min             2244
At 25th            2047
At 50th            1760
Below Min          1182
Above Market       1114
At 75th             511
At Max              303

Price Position Percentages:
price_position
No Market Data    83.77%
Below Market       5.71%
At Min             2.58%
At 25th            2.35%
At 50th            2.02%
Below Min          1.36%
Above Market       1.28%
At 75th            0.59%
At Max             0.35%
Name: proportion, dtype: object

Sample data with price position:


,product_id,warehouse_id,sku,current_price,discount_perc,price_after_discount,minimum,maximum,price_position
0,4940,797,مناديل فاميليا مطبخ كلاسيك 6 رول + 2 رول هدية ...,252.75,0.000000,252.750000,NaN,NaN,No Market Data
1,22318,632,اوريو بسكوت بكريمة بطعم الشوكولاتة والبندق - ...,90.25,0.000000,90.250000,100.0,102.0,Below Market
2,788,797,حلوانى مربى فراولة- 340 جم,383.00,0.000000,383.000000,399.0,410.0,Below Market
3,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,46.00,0.000000,46.000000,NaN,NaN,No Market Data
4,9786,236,عصير جهينة بيور جوافة كوكتيل - 235 مل,406.00,0.003100,404.741375,406.0,425.0,Below Min
5,9786,962,عصير جهينة بيور جوافة كوكتيل - 235 مل,406.00,0.003055,404.759720,406.0,425.0,Below Min
6,10922,339,المطبخ مكرونة لسان عصفور- 400 جم,207.00,0.000000,207.000000,NaN,NaN,No Market Data
7,10922,170,المطبخ مكرونة لسان عصفور- 400 جم,207.00,0.000000,207.000000,NaN,NaN,No Market Data
8,8635,703,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,85.00,0.011800,83.997000,83.5,90.0,At Min
9,362,1,عصير جهينة مانجو - 235 مل,209.50,0.000484,209.398640,208.8,215.0,At Min


In [20]:
# =============================================================================
# Stock Query - Get available stock by warehouse and product
# =============================================================================
STOCK_QUERY = '''
with parent_whs as (
select * 
from (
VALUES
(236,343),
(1,467),
(962,343)
)x(parent_id,child_id)

)
select warehouse_id,product_id,case when stocks_child is not null and stocks = 0 then stocks_child else stocks end as stocks 
from (
SELECT 
    pw.warehouse_id,
    pw.product_id,
    pw.available_stock::INTEGER AS stocks,
	pw2.available_stock::INTEGER AS stocks_child
	

FROM product_warehouse pw
left join parent_whs p on p.parent_id = pw.warehouse_id
left join product_warehouse pw2 on pw2.warehouse_id = p.child_id and pw.product_id = pw2.product_id
WHERE pw.warehouse_id NOT IN (6, 9, 10)
    AND pw.is_basic_unit = 1
)
'''

# Execute stock query
print("Loading stock data...")
df_stocks = query_snowflake(STOCK_QUERY)
df_stocks = convert_to_numeric(df_stocks)
print(f"Loaded {len(df_stocks)} stock records")

# Merge stock data with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_stocks[['warehouse_id', 'product_id', 'stocks']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing stocks with 0
pricing_with_discount['stocks'] = pricing_with_discount['stocks'].fillna(0).astype(int)

print(f"\nStock data merged!")
print(f"Records with stock (stocks > 0): {len(pricing_with_discount[pricing_with_discount['stocks'] > 0])}")
print(f"Records without stock (stocks = 0): {len(pricing_with_discount[pricing_with_discount['stocks'] == 0])}")
print(f"\nSample data with stocks:")
pricing_with_discount[
    ['product_id', 'warehouse_id', 'sku', 'stocks', 'price_after_discount', 'price_position']
].head(10)


Loading stock data...


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Loaded 1909646 stock records



Stock data merged!
Records with stock (stocks > 0): 22957
Records without stock (stocks = 0): 75708

Sample data with stocks:


,product_id,warehouse_id,sku,stocks,price_after_discount,price_position
0,4940,797,مناديل فاميليا مطبخ كلاسيك 6 رول + 2 رول هدية ...,0,252.750000,No Market Data
1,22318,632,اوريو بسكوت بكريمة بطعم الشوكولاتة والبندق - ...,8,90.250000,Below Market
2,788,797,حلوانى مربى فراولة- 340 جم,10,383.000000,Below Market
3,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,99,46.000000,No Market Data
4,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,99,46.000000,No Market Data
5,9786,236,عصير جهينة بيور جوافة كوكتيل - 235 مل,25,404.741375,Below Min
6,9786,962,عصير جهينة بيور جوافة كوكتيل - 235 مل,20,404.759720,Below Min
7,10922,339,المطبخ مكرونة لسان عصفور- 400 جم,0,207.000000,No Market Data
8,10922,170,المطبخ مكرونة لسان عصفور- 400 جم,0,207.000000,No Market Data
9,8635,703,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,39,83.997000,At Min


In [21]:
# =============================================================================
# Zero Demand Query - Identify SKUs with zero/low demand
# =============================================================================
ZERO_DEMAND_QUERY = f'''
WITH last_oss AS (
    SELECT product_id, warehouse_id, TIMESTAMP AS last_in_stock_day
    FROM (
        SELECT *, ROW_NUMBER() OVER(PARTITION BY product_id, warehouse_id ORDER BY TIMESTAMP DESC) AS rnk 
        FROM materialized_views.STOCK_DAY_CLOSE
        WHERE AVAILABLE_STOCK = 0 
            AND TIMESTAMP >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
        QUALIFY rnk = 1 
    )
),

current_stocks AS (
    SELECT product_id, warehouse_id, AVAILABLE_STOCK, activation
    FROM PRODUCT_WAREHOUSE
    WHERE IS_BASIC_UNIT = 1
        AND CASE WHEN product_id = 1309 THEN packing_unit_id <> 23 ELSE TRUE END
),

prs AS (
    SELECT DISTINCT 
        product_purchased_receipts.product_id,
        purchased_receipts.warehouse_id,
        purchased_receipts.date::DATE AS date,
        product_purchased_receipts.purchased_item_count * product_purchased_receipts.basic_unit_count AS purchase_min_count
    FROM product_purchased_receipts
    JOIN purchased_receipts ON purchased_receipts.id = product_purchased_receipts.purchased_receipt_id
    JOIN last_oss lo ON product_purchased_receipts.product_id = lo.product_id 
        AND lo.warehouse_id = purchased_receipts.warehouse_id 
        AND purchased_receipts.date > lo.last_in_stock_day 
    WHERE product_purchased_receipts.purchased_item_count <> 0
        AND purchased_receipts.purchased_receipt_status_id IN (4, 5, 7)
        AND purchased_receipts.date::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
),

main AS (
    SELECT 
        prs.product_id, 
        prs.warehouse_id, 
        MIN(date) AS first_order_date, 
        SUM(purchase_min_count) AS total_recieved, 
        cs.AVAILABLE_STOCK, 
        cs.activation
    FROM prs 
    JOIN current_stocks cs ON cs.product_id = prs.product_id AND prs.warehouse_id = cs.warehouse_id
    GROUP BY prs.product_id, prs.warehouse_id, cs.AVAILABLE_STOCK, cs.activation
),

sold_days AS (
    SELECT product_id, warehouse_id, COUNT(DISTINCT o_date) AS sales_days
    FROM (
        SELECT DISTINCT
            so.created_at::DATE AS o_date,
            pso.warehouse_id,
            pso.product_id,
            SUM(pso.purchased_item_count * basic_unit_count) AS daily_qty
        FROM product_sales_order pso
        JOIN sales_orders so ON so.id = pso.sales_order_id
        JOIN main m ON m.product_id = pso.product_id 
            AND m.warehouse_id = pso.warehouse_id 
            AND so.created_at::DATE >= m.first_order_date
        WHERE so.created_at::DATE BETWEEN CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120 
            AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
            AND so.sales_order_status_id NOT IN (7, 12)
            AND so.channel IN ('telesales', 'retailer')
            AND pso.purchased_item_count <> 0
        GROUP BY o_date, pso.warehouse_id, pso.product_id
    )
    GROUP BY product_id, warehouse_id
)

SELECT DISTINCT warehouse_id, product_id
FROM (
    SELECT m.product_id, m.warehouse_id, m.first_order_date, m.activation,
        COALESCE(sd.sales_days, 0) AS sales_days,
        COALESCE(sd.sales_days, 0)::FLOAT / NULLIF((CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1) - m.first_order_date, 0) AS perc_days
    FROM main m 
    LEFT JOIN sold_days sd ON sd.product_id = m.product_id AND sd.warehouse_id = m.warehouse_id
    WHERE m.first_order_date < CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 10
)
WHERE perc_days <= 0.3
    AND activation = 'true'
'''

# Execute zero demand query
print("Loading zero demand SKUs...")
df_zero_demand = query_snowflake(ZERO_DEMAND_QUERY)
df_zero_demand = convert_to_numeric(df_zero_demand)
print(f"Loaded {len(df_zero_demand)} zero demand SKU records")


Loading zero demand SKUs...


Loaded 2610 zero demand SKU records


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [22]:
# =============================================================================
# Add Zero Demand Flag to pricing_with_discount
# =============================================================================

# Add a marker column to identify zero demand SKUs
df_zero_demand['zero_demand'] = 1

# Merge with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_zero_demand[['warehouse_id', 'product_id', 'zero_demand']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing values with 0 (not zero demand)
pricing_with_discount['zero_demand'] = pricing_with_discount['zero_demand'].fillna(0).astype(int)

print(f"Zero demand flag added!")
print(f"SKUs flagged as zero demand: {len(pricing_with_discount[pricing_with_discount['zero_demand'] == 1])}")
print(f"SKUs with normal demand: {len(pricing_with_discount[pricing_with_discount['zero_demand'] == 0])}")


Zero demand flag added!
SKUs flagged as zero demand: 2769
SKUs with normal demand: 95896


In [23]:
# =============================================================================
# OOS Yesterday Query - Identify SKUs out of stock yesterday
# =============================================================================
OOS_YESTERDAY_QUERY = f'''
SELECT DISTINCT product_id, warehouse_id,
    CASE WHEN opening_stocks = 0 AND closing_stocks = 0 THEN 1
         ELSE 0 
    END AS oos_yesterday
FROM (
    SELECT 
        timestamp,
        product_id,
        warehouse_id, 
        AVAILABLE_STOCK AS closing_stocks,
        LAG(AVAILABLE_STOCK) OVER (PARTITION BY product_id, warehouse_id ORDER BY TIMESTAMP) AS opening_stocks
    FROM materialized_views.stock_day_close
    WHERE timestamp::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 2
    QUALIFY opening_stocks IS NOT NULL
)
WHERE oos_yesterday = 1
'''

# Execute OOS yesterday query
print("Loading OOS yesterday data...")
df_oos_yesterday = query_snowflake(OOS_YESTERDAY_QUERY)
df_oos_yesterday = convert_to_numeric(df_oos_yesterday)
print(f"Loaded {len(df_oos_yesterday)} OOS yesterday records")


Loading OOS yesterday data...


Loaded 1931259 OOS yesterday records


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [24]:
# =============================================================================
# Add OOS Yesterday Flag to pricing_with_discount
# =============================================================================

# Merge with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_oos_yesterday[['warehouse_id', 'product_id', 'oos_yesterday']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing values with 0 (not OOS yesterday)
pricing_with_discount['oos_yesterday'] = pricing_with_discount['oos_yesterday'].fillna(0).astype(int)

print(f"OOS yesterday flag added!")
print(f"SKUs out of stock yesterday: {len(pricing_with_discount[pricing_with_discount['oos_yesterday'] == 1])}")
print(f"SKUs in stock yesterday: {len(pricing_with_discount[pricing_with_discount['oos_yesterday'] == 0])}")


OOS yesterday flag added!
SKUs out of stock yesterday: 74809
SKUs in stock yesterday: 23856


In [25]:
# =============================================================================
# Running Rate Query - SKU x Warehouse daily forecast (1-day forecast)
# Enhanced with ABC classification, zero demand ratio, and recently received detection
# =============================================================================
RUNNING_RATE_QUERY = f'''
WITH params AS (
  SELECT
    CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE AS run_date,
    DATEADD(month, -3, CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) AS history_start,
    DATEADD(month, -2, CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) AS history_start_zero_ratio,
    21 AS recency_half_life_days,
    4 AS zero_rule_days,
    5 AS min_days_threshold,
    0.7 AS zero_demand_ratio_threshold,
    14 AS recent_receipt_days
),

/* 1) Daily sales aggregation - 3 months */
sales_base AS (
  SELECT
    pso.product_id AS product_id,
    pso.warehouse_id,
    CAST(DATE_TRUNC('day', pso.created_at) AS DATE) AS date,
    SUM(pso.purchased_item_count * pso.basic_unit_count) AS sold_units,
    SUM(pso.purchased_item_count * pso.basic_unit_count * pso.item_price)
      / NULLIF(SUM(pso.purchased_item_count * pso.basic_unit_count), 0) AS avg_selling_price,
    COUNT(DISTINCT so.retailer_id) AS retailer_count
  FROM product_sales_order pso
  JOIN sales_orders so ON pso.sales_order_id = so.id
  WHERE CAST(DATE_TRUNC('day', pso.created_at) AS DATE) >= (SELECT history_start FROM params)
    AND so.sales_order_status_id NOT IN (7, 12)
    AND pso.purchased_item_count <> 0
  GROUP BY 1, 2, 3
),

/* 2a) Primary stock source - STOCK_DAY_CLOSE (fast) */
stock_day_close AS (
  SELECT
    dc.product_id,
    dc.warehouse_id,
    CAST(DATE_TRUNC('day', dc.timestamp) AS DATE) AS date,
    MAX(dc.available_stock) AS stock_closing,
    CASE WHEN MAX(dc.available_stock) > 0 THEN 1 ELSE 0 END AS in_stock_flag,
    0 AS oos_hours
  FROM materialized_views.STOCK_DAY_CLOSE dc
  WHERE dc.product_id IS NOT NULL
    AND CAST(DATE_TRUNC('day', dc.timestamp) AS DATE) >= (SELECT history_start FROM params)
    AND CAST(DATE_TRUNC('day', dc.timestamp) AS DATE) < CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
  GROUP BY dc.product_id, dc.warehouse_id, CAST(DATE_TRUNC('day', dc.timestamp) AS DATE)
),

/* 2b) Identify days where closing stock = 0 BUT sales existed */
sales_with_zero_close AS (
  SELECT sb.product_id, sb.warehouse_id, sb.date
  FROM sales_base sb
  JOIN stock_day_close dc ON sb.product_id = dc.product_id AND sb.warehouse_id = dc.warehouse_id AND sb.date = dc.date
  WHERE sb.sold_units > 0 AND dc.in_stock_flag = 0
),

/* 2c) Drill into stock_snap_shots_recent ONLY for edge case days */
hourly_edge_cases AS (
  SELECT
    ss.product_id,
    ss.warehouse_id,
    CAST(DATE_TRUNC('day', ss.timestamp) AS DATE) AS date,
    CASE WHEN MAX(CASE WHEN ss.activation = TRUE AND ss.available_stock > 0 THEN 1 ELSE 0 END) = 1 THEN 1 ELSE 0 END AS had_midday_stock,
    24 * (SUM(CASE WHEN ss.activation = FALSE OR ss.available_stock = 0 THEN 1 ELSE 0 END)::FLOAT / NULLIF(COUNT(*), 0)) AS oos_hours
  FROM materialized_views.stock_snap_shots_recent ss
  JOIN sales_with_zero_close sc ON ss.product_id = sc.product_id AND ss.warehouse_id = sc.warehouse_id AND CAST(DATE_TRUNC('day', ss.timestamp) AS DATE) = sc.date
  WHERE ss.product_id IS NOT NULL
  GROUP BY ss.product_id, ss.warehouse_id, CAST(DATE_TRUNC('day', ss.timestamp) AS DATE)
),

/* 2d) SKUs with sales in the last 7 days - relaxed in_stock_flag */
recent_sales_skus AS (
  SELECT DISTINCT pso.product_id, pso.warehouse_id
  FROM product_sales_order pso
  JOIN sales_orders so ON pso.sales_order_id = so.id
  WHERE CAST(DATE_TRUNC('day', pso.created_at) AS DATE) >= DATEADD(day, -7, CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
    AND so.sales_order_status_id NOT IN (7, 12)
    AND pso.purchased_item_count <> 0
),

/* 2e) All days where sales occurred - sales proof override */
sales_days AS (
  SELECT DISTINCT pso.product_id, pso.warehouse_id, CAST(DATE_TRUNC('day', pso.created_at) AS DATE) AS date
  FROM product_sales_order pso
  JOIN sales_orders so ON pso.sales_order_id = so.id
  WHERE CAST(DATE_TRUNC('day', pso.created_at) AS DATE) >= (SELECT history_start FROM params)
    AND so.sales_order_status_id NOT IN (7, 12)
    AND pso.purchased_item_count <> 0
),

/* 2f) Final stock: merge STOCK_DAY_CLOSE with hourly edge case overrides */
stock_daily_final AS (
  SELECT
    dc.product_id,
    dc.warehouse_id,
    dc.date,
    dc.stock_closing,
    CASE
      WHEN sal.product_id IS NOT NULL THEN 0
      WHEN ec.product_id IS NOT NULL THEN ec.oos_hours
      ELSE dc.oos_hours
    END AS oos_hours,
    CASE
      WHEN sal.product_id IS NOT NULL THEN 1
      WHEN ec.product_id IS NOT NULL THEN ec.had_midday_stock
      WHEN rs.product_id IS NOT NULL THEN dc.in_stock_flag
      ELSE dc.in_stock_flag
    END AS in_stock_flag
  FROM stock_day_close dc
  LEFT JOIN sales_days sal ON dc.product_id = sal.product_id AND dc.warehouse_id = sal.warehouse_id AND dc.date = sal.date
  LEFT JOIN hourly_edge_cases ec ON dc.product_id = ec.product_id AND dc.warehouse_id = ec.warehouse_id AND dc.date = ec.date
  LEFT JOIN recent_sales_skus rs ON dc.product_id = rs.product_id AND dc.warehouse_id = rs.warehouse_id
),

/* 3) Join sales + stock */
base_data AS (
  SELECT
    sd.product_id,
    sd.warehouse_id,
    sd.date,
    COALESCE(sb.sold_units, 0) AS sold_units,
    sb.avg_selling_price,
    COALESCE(sb.retailer_count, 0) AS retailer_count,
    sd.stock_closing,
    sd.oos_hours,
    sd.in_stock_flag,
    CASE WHEN DAYOFWEEKISO(sd.date) IN (5, 6) THEN 1 ELSE 0 END AS is_weekend
  FROM stock_daily_final sd
  LEFT JOIN sales_base sb ON sd.product_id = sb.product_id AND sd.warehouse_id = sb.warehouse_id AND sd.date = sb.date
  WHERE sd.in_stock_flag = 1
  UNION
  SELECT
    sb.product_id,
    sb.warehouse_id,
    sb.date,
    sb.sold_units,
    sb.avg_selling_price,
    sb.retailer_count,
    NULL AS stock_closing,
    0 AS oos_hours,
    1 AS in_stock_flag,
    CASE WHEN DAYOFWEEKISO(sb.date) IN (5, 6) THEN 1 ELSE 0 END AS is_weekend
  FROM sales_base sb
  LEFT JOIN stock_daily_final sd ON sb.product_id = sd.product_id AND sb.warehouse_id = sd.warehouse_id AND sb.date = sd.date
  WHERE sd.product_id IS NULL
),

/* 4) Stats per SKU x WH */
sku_wh_stats AS (
  SELECT
    product_id,
    warehouse_id,
    AVG(sold_units) AS avg_units,
    STDDEV_SAMP(sold_units) AS sigma_d,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY sold_units) AS med_units,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY sold_units) AS pct95_units,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY retailer_count) AS med_retailers
  FROM base_data
  GROUP BY product_id, warehouse_id
),

/* 5) ABC classification */
abc_class AS (
  SELECT product_id, warehouse_id,
    CASE
      WHEN cumulative_sales_pct <= 0.80 THEN 'A'
      WHEN cumulative_sales_pct <= 0.95 THEN 'B'
      ELSE 'C'
    END AS abc_class
  FROM (
    SELECT product_id, warehouse_id,
      SUM(sold_units * avg_selling_price) AS sales_value,
      SUM(SUM(sold_units * avg_selling_price)) OVER (PARTITION BY warehouse_id ORDER BY SUM(sold_units * avg_selling_price) DESC)
        / NULLIF(SUM(SUM(sold_units * avg_selling_price)) OVER (PARTITION BY warehouse_id), 0) AS cumulative_sales_pct
    FROM base_data
    GROUP BY product_id, warehouse_id
  ) t
  QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id, warehouse_id ORDER BY cumulative_sales_pct ASC) = 1
),

/* 6) Remove outliers */
cleaned AS (
  SELECT b.*, s.med_units, s.pct95_units, s.med_retailers,
    CASE WHEN b.sold_units > s.pct95_units THEN s.pct95_units ELSE b.sold_units END AS units_capped,
    CASE WHEN b.retailer_count > GREATEST(2, s.med_retailers * 2) THEN 1 ELSE 0 END AS retailer_spike
  FROM base_data b
  LEFT JOIN sku_wh_stats s ON b.product_id = s.product_id AND b.warehouse_id = s.warehouse_id
),

/* 7) Scale down retailer spikes */
adjusted AS (
  SELECT *,
    CASE
      WHEN retailer_spike = 1 AND retailer_count > 0 AND med_retailers IS NOT NULL
        THEN ROUND(units_capped * (med_retailers::FLOAT / NULLIF(retailer_count::FLOAT, 0)), 0)
      ELSE units_capped
    END AS units_adjusted
  FROM cleaned
),

/* 8) Add weighting */
weighted AS (
  SELECT a.*,
    DATEDIFF('day', a.date, (SELECT run_date FROM params)) AS days_ago,
    CASE
      WHEN a.date >= DATEADD(day, -21, (SELECT run_date FROM params)) THEN 1.5
      WHEN a.date >= DATEADD(day, -90, (SELECT run_date FROM params)) THEN 1.0
      ELSE 0.5
    END AS w_recency,
    CASE
      WHEN COALESCE(a.in_stock_flag, 0) = 1 AND COALESCE(a.oos_hours, 0) < 12 THEN 1.4
      WHEN COALESCE(a.in_stock_flag, 0) = 1 AND COALESCE(a.oos_hours, 0) >= 12 THEN 0.9
      ELSE 0.6
    END AS w_instock,
    CASE WHEN a.is_weekend = 1 THEN 0.7 ELSE 1.0 END AS w_weekend
  FROM adjusted a
),

/* 9) Weighted final rows */
weighted_final AS (
  SELECT w.product_id, w.warehouse_id, abc.abc_class, w.date, w.units_adjusted, w.oos_hours, w.in_stock_flag,
    (w.w_recency * w.w_instock * w.w_weekend) AS final_weight
  FROM weighted w
  LEFT JOIN abc_class abc ON w.product_id = abc.product_id AND w.warehouse_id = abc.warehouse_id
  WHERE w.units_adjusted IS NOT NULL AND w.date >= (SELECT history_start FROM params)
),

/* 10) Forecast base */
forecast_base AS (
  SELECT wf.product_id, wf.warehouse_id, abc.abc_class,
    SUM(wf.units_adjusted * wf.final_weight) / NULLIF(SUM(wf.final_weight), 0) AS weighted_avg_units,
    COUNT(*) AS n_days_used,
    FALSE AS used_extended_lookback
  FROM weighted_final wf
  LEFT JOIN abc_class abc ON wf.product_id = abc.product_id AND wf.warehouse_id = abc.warehouse_id
  GROUP BY wf.product_id, wf.warehouse_id, abc.abc_class
),

/* 11) Zero-sales last 4 days detection */
last_4_days AS (
  SELECT sd.product_id, sd.warehouse_id, sd.date, COALESCE(sb.sold_units, 0) AS sold_units, 1 AS in_stock_flag
  FROM stock_daily_final sd
  LEFT JOIN sales_base sb ON sd.product_id = sb.product_id AND sd.warehouse_id = sb.warehouse_id AND sd.date = sb.date
  WHERE sd.date >= DATEADD(day, -4, (SELECT run_date FROM params)) AND sd.date < (SELECT run_date FROM params) AND sd.in_stock_flag = 1
  UNION
  SELECT sb.product_id, sb.warehouse_id, sb.date, sb.sold_units, 1 AS in_stock_flag
  FROM sales_base sb
  LEFT JOIN stock_daily_final sd ON sb.product_id = sd.product_id AND sb.warehouse_id = sd.warehouse_id AND sb.date = sd.date
  WHERE sb.date >= DATEADD(day, -4, (SELECT run_date FROM params)) AND sb.date < (SELECT run_date FROM params)
    AND sb.sold_units > 0 AND (sd.product_id IS NULL OR sd.in_stock_flag = 0)
),

last4_flag AS (
  SELECT product_id, warehouse_id,
    CASE WHEN COUNT(*) = 4 AND SUM(CASE WHEN COALESCE(sold_units, 0) = 0 AND COALESCE(in_stock_flag, 0) = 1 THEN 1 ELSE 0 END) = 4
    THEN 1 ELSE 0 END AS last4_all_instock_zero
  FROM last_4_days
  GROUP BY product_id, warehouse_id
),

/* 12) Class C zero-demand ratio - past 2 months only */
class_c_zero_ratio AS (
  SELECT sd.product_id, sd.warehouse_id,
    COUNT(*) AS total_instock_days,
    SUM(CASE WHEN COALESCE(sb.sold_units, 0) = 0 THEN 1 ELSE 0 END) AS zero_sales_days,
    SUM(CASE WHEN COALESCE(sb.sold_units, 0) = 0 THEN 1 ELSE 0 END)::FLOAT / NULLIF(COUNT(*), 0) AS zero_demand_ratio
  FROM stock_daily_final sd
  LEFT JOIN sales_base sb ON sd.product_id = sb.product_id AND sd.warehouse_id = sb.warehouse_id AND sd.date = sb.date
  WHERE sd.date >= (SELECT history_start_zero_ratio FROM params) AND sd.in_stock_flag = 1
  GROUP BY sd.product_id, sd.warehouse_id
),

first_sale AS (
  SELECT product_id, warehouse_id, MIN(date) AS first_sale_date
  FROM base_data WHERE sold_units > 0
  GROUP BY product_id, warehouse_id
),

/* 13) Recently received SKUs - last 14 days */
recent_receipts AS (
  SELECT DISTINCT ppr.product_id, pr.warehouse_id
  FROM product_purchased_receipts ppr
  JOIN purchased_receipts pr ON pr.id = ppr.purchased_receipt_id
  WHERE pr.purchased_receipt_status_id IN (4, 5, 7)
    AND ppr.purchased_item_count <> 0
    AND pr.date::DATE >= DATEADD(day, -14, CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
),

/* 14) Final forecast */
final_forecast AS (
  SELECT fb.product_id, fb.warehouse_id, fb.abc_class, fb.weighted_avg_units, fb.n_days_used, fb.used_extended_lookback, s.med_units,
    CASE
      WHEN rr.product_id IS NOT NULL AND fb.n_days_used >= (SELECT min_days_threshold FROM params)
        THEN CASE WHEN fb.abc_class = 'C'
          THEN LEAST(CEIL(fb.weighted_avg_units), CEIL(CASE WHEN COALESCE(s.med_units, 0) > 0 THEN s.med_units ELSE fb.weighted_avg_units END))
          ELSE CEIL(fb.weighted_avg_units) END
      WHEN rr.product_id IS NOT NULL AND fb.n_days_used < (SELECT min_days_threshold FROM params)
        THEN GREATEST(CEIL(fb.weighted_avg_units), 1)
      WHEN l4.last4_all_instock_zero = 1 AND fb.abc_class = 'C' THEN 0
      WHEN fb.abc_class = 'C' AND COALESCE(czr.zero_demand_ratio, 1) > (SELECT zero_demand_ratio_threshold FROM params) THEN 0
      WHEN fb.n_days_used < (SELECT min_days_threshold FROM params) AND COALESCE(czr.zero_demand_ratio, 1) = 0 AND czr.total_instock_days > 0
        THEN GREATEST(CEIL(fb.weighted_avg_units), 1)
      WHEN fb.n_days_used < (SELECT min_days_threshold FROM params) THEN 0
      WHEN fb.abc_class = 'C' AND COALESCE(czr.zero_demand_ratio, 1) <= (SELECT zero_demand_ratio_threshold FROM params) AND fb.n_days_used < (SELECT min_days_threshold FROM params) THEN 1
      WHEN fs.first_sale_date IS NOT NULL AND fs.first_sale_date >= DATEADD(day, -2, (SELECT run_date FROM params))
        THEN GREATEST(CEIL(fb.weighted_avg_units), 1)
      WHEN fb.abc_class = 'C'
        THEN LEAST(CEIL(fb.weighted_avg_units), CEIL(CASE WHEN COALESCE(s.med_units, 0) > 0 THEN s.med_units ELSE fb.weighted_avg_units END))
      ELSE CEIL(fb.weighted_avg_units)
    END AS in_stock_rr
  FROM forecast_base fb
  LEFT JOIN last4_flag l4 ON fb.product_id = l4.product_id AND fb.warehouse_id = l4.warehouse_id
  LEFT JOIN class_c_zero_ratio czr ON fb.product_id = czr.product_id AND fb.warehouse_id = czr.warehouse_id
  LEFT JOIN first_sale fs ON fb.product_id = fs.product_id AND fb.warehouse_id = fs.warehouse_id
  LEFT JOIN sku_wh_stats s ON fb.product_id = s.product_id AND fb.warehouse_id = s.warehouse_id
  LEFT JOIN recent_receipts rr ON fb.product_id = rr.product_id AND fb.warehouse_id = rr.warehouse_id
)

SELECT
  ff.product_id,
  ff.warehouse_id,
  ff.in_stock_rr,
  s.sigma_d,
  czr.zero_demand_ratio,
  CASE WHEN rr.product_id IS NOT NULL THEN TRUE ELSE FALSE END AS recently_received
FROM final_forecast ff
LEFT JOIN sku_wh_stats s ON ff.product_id = s.product_id AND ff.warehouse_id = s.warehouse_id
LEFT JOIN class_c_zero_ratio czr ON ff.product_id = czr.product_id AND ff.warehouse_id = czr.warehouse_id
LEFT JOIN recent_receipts rr ON ff.product_id = rr.product_id AND ff.warehouse_id = rr.warehouse_id
'''

# Execute running rate query
print("Loading running rate data (this may take a moment)...")
df_running_rate = query_snowflake(RUNNING_RATE_QUERY)
df_running_rate = convert_to_numeric(df_running_rate)
print(f"Loaded {len(df_running_rate)} running rate records")


Loading running rate data (this may take a moment)...


Loaded 32123 running rate records


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [26]:
# =============================================================================
# Merge Running Rate and Calculate DOH (Days on Hand)
# =============================================================================

# Merge running rate data with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_running_rate[['warehouse_id', 'product_id', 'in_stock_rr']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing values for running rate columns
pricing_with_discount['in_stock_rr'] = pricing_with_discount['in_stock_rr'].fillna(0)

# Calculate DOH (Days on Hand) = stocks / in_stock_rr
# Handle division by zero - if running rate is 0, DOH is infinite (use 999)
pricing_with_discount['doh'] = np.select(
    [
        (pricing_with_discount['in_stock_rr'] > 0) & (pricing_with_discount['stocks'] > 0),
        pricing_with_discount['stocks'] == 0
    ],
    [
        pricing_with_discount['stocks'] / pricing_with_discount['in_stock_rr'],
        0
    ],
    default=999
)


In [27]:
# =============================================================================
# Product Classification Query - ABC Classification based on order contribution
# =============================================================================
PRODUCT_CLASSIFICATION_QUERY = f'''
WITH order_counts AS (
    SELECT 
        pso.warehouse_id,
        pso.product_id,
        COUNT(DISTINCT pso.sales_order_id) AS order_count
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 90
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY pso.warehouse_id, pso.product_id
),

warehouse_totals AS (
    SELECT 
        warehouse_id,
        SUM(order_count) AS total_orders
    FROM order_counts
    GROUP BY warehouse_id
),

ranked_products AS (
    SELECT 
        oc.warehouse_id,
        oc.product_id,
        oc.order_count,
        wt.total_orders,
        oc.order_count::FLOAT / NULLIF(wt.total_orders, 0) AS contribution,
        SUM(oc.order_count::FLOAT / NULLIF(wt.total_orders, 0)) 
            OVER (PARTITION BY oc.warehouse_id ORDER BY oc.order_count DESC 
                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_contribution
    FROM order_counts oc
    JOIN warehouse_totals wt ON oc.warehouse_id = wt.warehouse_id
)

SELECT 
    warehouse_id,
    product_id,
    order_count,
    contribution,
    cumulative_contribution,
    CASE 
        WHEN cumulative_contribution <= 0.3 THEN 'A'
        WHEN cumulative_contribution <= 0.75 THEN 'B'
        ELSE 'C'
    END AS abc_class
FROM ranked_products
'''

# Execute product classification query
print("Loading product classification data...")
df_classification = query_snowflake(PRODUCT_CLASSIFICATION_QUERY)
df_classification = convert_to_numeric(df_classification)
print(f"Loaded {len(df_classification)} product classification records")
print(f"\nClassification distribution:")
print(df_classification['abc_class'].value_counts().to_string())


Loading product classification data...


Loaded 28273 product classification records

Classification distribution:
abc_class
C    22543
B     4975
A      755


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [28]:
# =============================================================================
# Add ABC Classification to pricing_with_discount
# =============================================================================

# Merge classification data with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_classification[['warehouse_id', 'product_id', 'order_count', 'contribution', 'abc_class']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing values - products without orders in last 3 months get class 'C'
pricing_with_discount['order_count'] = pricing_with_discount['order_count'].fillna(0).astype(int)
pricing_with_discount['contribution'] = pricing_with_discount['contribution'].fillna(0)
pricing_with_discount['abc_class'] = pricing_with_discount['abc_class'].fillna('C')

print(f"ABC Classification added!")
print(f"\nClassification in pricing_with_discount:")
print(pricing_with_discount['abc_class'].value_counts().to_string())
print(f"\nSample data with classification:")
pricing_with_discount[
    ['product_id', 'warehouse_id', 'sku', 'order_count', 'contribution', 'abc_class', 'stocks', 'doh']
].head(15)


ABC Classification added!

Classification in pricing_with_discount:
abc_class
C    92375
B     5500
A      790

Sample data with classification:


,product_id,warehouse_id,sku,order_count,contribution,abc_class,stocks,doh
0,4940,797,مناديل فاميليا مطبخ كلاسيك 6 رول + 2 رول هدية ...,148,0.000972,B,0,0.000000
1,22318,632,اوريو بسكوت بكريمة بطعم الشوكولاتة والبندق - ...,19,0.000200,C,8,8.000000
2,788,797,حلوانى مربى فراولة- 340 جم,5,0.000033,C,10,10.000000
3,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,514,0.000604,B,99,5.823529
4,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,514,0.000604,B,99,5.823529
5,9786,236,عصير جهينة بيور جوافة كوكتيل - 235 مل,187,0.000349,C,25,12.500000
6,9786,962,عصير جهينة بيور جوافة كوكتيل - 235 مل,315,0.000611,B,20,5.000000
7,10922,339,المطبخ مكرونة لسان عصفور- 400 جم,0,0.000000,C,0,0.000000
8,10922,170,المطبخ مكرونة لسان عصفور- 400 جم,0,0.000000,C,0,0.000000
9,8635,703,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,207,0.001213,B,39,7.800000


In [29]:
# =============================================================================
# PO (Purchase Order) Data Query - Last PO status and rejection count
# =============================================================================
PO_DATA_QUERY = '''
WITH last_data AS (
    SELECT product_id, warehouse_id, confirmation_status, PO_date::DATE AS last_po_date, ordered_qty
    FROM (
        SELECT 
            product_id,
            Target_WAREHOUSE_ID AS warehouse_id,
            confirmation_status,
            created_at AS PO_date,
            MIN_QUANTITY AS ordered_qty,
            reason,
            MAX(created_at) OVER (PARTITION BY product_id, Target_WAREHOUSE_ID) AS last_po
        FROM retool.PO_INITIAL_PLAN
        WHERE created_at::DATE >= CURRENT_DATE - 15 
    ) x
    WHERE last_po = PO_date
),

last_15_data AS (
    SELECT 
        product_id,
        target_WAREHOUSE_ID AS warehouse_id,
        COUNT(DISTINCT CASE WHEN confirmation_status <> 'yes' THEN created_at END) AS no_last_15
    FROM retool.PO_INITIAL_PLAN
    WHERE created_at::DATE >= CURRENT_DATE - 15 
    GROUP BY 1, 2
)

SELECT 
    ld.product_id,
    ld.warehouse_id,
    ld.confirmation_status,
    ld.last_po_date,
    ld.ordered_qty,
    COALESCE(lfd.no_last_15, 0) AS no_last_15
FROM last_data ld 
LEFT JOIN last_15_data lfd 
    ON lfd.product_id = ld.product_id 
    AND lfd.warehouse_id = ld.warehouse_id
'''

# Execute PO data query using dwh_pg_query
print("Loading PO data...")
df_po_data = setup_environment_2.dwh_pg_query(
    PO_DATA_QUERY, 
    columns=['product_id', 'warehouse_id', 'confirmation_status', 'last_po_date', 'ordered_qty', 'no_last_15']
)
df_po_data.columns = df_po_data.columns.str.lower()
df_po_data = convert_to_numeric(df_po_data)
print(f"Loaded {len(df_po_data)} PO records")
print(f"\nConfirmation status distribution:")
print(df_po_data['confirmation_status'].value_counts().to_string())


Loading PO data...


Loaded 18524 PO records

Confirmation status distribution:
confirmation_status
yes    14917
no      3150


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [30]:
# =============================================================================
# Add PO Data to pricing_with_discount
# =============================================================================

# Merge PO data with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_po_data[['warehouse_id', 'product_id', 'confirmation_status', 'last_po_date', 'ordered_qty', 'no_last_15']], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill missing values
pricing_with_discount['ordered_qty'] = pricing_with_discount['ordered_qty'].fillna(0)
pricing_with_discount['no_last_15'] = pricing_with_discount['no_last_15'].fillna(0).astype(int)

print(f"PO data added!")
print(f"\nRecords with PO data: {len(pricing_with_discount[~pricing_with_discount['confirmation_status'].isna()])}")
print(f"Records without PO data: {len(pricing_with_discount[pricing_with_discount['confirmation_status'].isna()])}")
print(f"\nSample data with PO info:")
pricing_with_discount[
    ['product_id', 'warehouse_id', 'sku', 'confirmation_status', 'last_po_date', 'ordered_qty', 'no_last_15']
].dropna(subset=['confirmation_status']).head(15)


PO data added!

Records with PO data: 20509
Records without PO data: 78156

Sample data with PO info:


,product_id,warehouse_id,sku,confirmation_status,last_po_date,ordered_qty,no_last_15
0,4940,797,مناديل فاميليا مطبخ كلاسيك 6 رول + 2 رول هدية ...,no,2026-02-25,42.0,10
1,22318,632,اوريو بسكوت بكريمة بطعم الشوكولاتة والبندق - ...,yes,2026-02-23,8.0,0
2,788,797,حلوانى مربى فراولة- 340 جم,yes,2026-02-23,4.0,0
3,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,yes,2026-02-25,20.0,0
4,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,yes,2026-02-25,20.0,0
5,9786,236,عصير جهينة بيور جوافة كوكتيل - 235 مل,yes,2026-02-22,8.0,0
6,9786,962,عصير جهينة بيور جوافة كوكتيل - 235 مل,yes,2026-02-25,8.0,0
9,8635,703,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,yes,2026-02-21,40.0,0
10,362,1,عصير جهينة مانجو - 235 مل,yes,2026-02-23,330.0,0
11,3889,797,عصير سن توب توت مشكل - 250 مل,yes,2026-02-23,34.0,0


In [31]:
# =============================================================================
# Leadtime Query - Supplier leadtime by brand, category, and warehouse
# =============================================================================
LEADTIME_QUERY = '''
SELECT brand, cat, warehouse_id, leadtime
FROM (
    SELECT a.*, b.name_ar AS brand, c.name_ar AS cat
    FROM (
        SELECT DISTINCT 
            sl.supplier_id, 
            warehouse_id, 
            category_id, 
            brand_id, 
            sl.updated_at, 
            leadtime,
            MAX(sl.updated_at) OVER (PARTITION BY sl.supplier_id, warehouse_id) AS last_update
        FROM retool.SUPPLIER_MOQ sl 
        JOIN retool.PO_SUPPLIER_MAPPING sm ON sl.supplier_id = sm.supplier_id 
    ) a
    JOIN brands b ON b.id = a.brand_id 
    JOIN categories c ON c.id = a.category_id
    WHERE a.updated_at = last_update
) d
'''

# Execute leadtime query using dwh_pg_query
print("Loading leadtime data...")
df_leadtime = setup_environment_2.dwh_pg_query(
    LEADTIME_QUERY, 
    columns=['brand', 'cat', 'warehouse_id', 'leadtime']
)
df_leadtime.columns = df_leadtime.columns.str.lower()
df_leadtime = convert_to_numeric(df_leadtime)
print(f"Loaded {len(df_leadtime)} leadtime records")


Loading leadtime data...


Loaded 15133 leadtime records


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [32]:
# =============================================================================
# Add Leadtime to pricing_with_discount
# =============================================================================

# Merge leadtime data with pricing_with_discount (by brand, cat, warehouse_id)
pricing_with_discount = pricing_with_discount.merge(
    df_leadtime[['brand', 'cat', 'warehouse_id', 'leadtime']], 
    on=['brand', 'cat', 'warehouse_id'], 
    how='left'
)

# Fill missing leadtime with 0 or a default value
pricing_with_discount['leadtime'] = pricing_with_discount['leadtime'].fillna(72)


print(f"Leadtime data added!")
print(f"\nRecords with leadtime: {len(pricing_with_discount[pricing_with_discount['leadtime'] > 0])}")
print(f"Records without leadtime: {len(pricing_with_discount[pricing_with_discount['leadtime'] == 0])}")
print(f"\nLeadtime distribution:")
print(pricing_with_discount['leadtime'].describe())

# =============================================================================
# Calculate Expected Receiving Day
# If confirmation_status is 'no': add 2 extra days (48 hours) before adding leadtime
# expected_receiving_day = last_po_date + ((2 + leadtime) / 24) if not confirmed
# expected_receiving_day = last_po_date + (leadtime / 24) if confirmed
# =============================================================================

# Convert last_po_date to datetime if not already
pricing_with_discount['last_po_date'] = pd.to_datetime(pricing_with_discount['last_po_date'], errors='coerce')

# Calculate adjusted leadtime: add 48 hours (2 days) if confirmation_status is 'no'
pricing_with_discount['adjusted_leadtime'] = np.where(
    pricing_with_discount['confirmation_status'].str.lower() == 'no',
    pricing_with_discount['leadtime'] + 48,  # Add 2 days (48 hours) if not confirmed
    pricing_with_discount['leadtime']
)

# Calculate expected receiving day (leadtime is in hours, divide by 24 for days)
pricing_with_discount['expected_receiving_day'] = pricing_with_discount['last_po_date'] + pd.to_timedelta(
    pricing_with_discount['adjusted_leadtime'] / 24, unit='D'
)

# Set expected_receiving_day to empty if it's in the past (smaller than today)
pricing_with_discount['expected_receiving_day'] = np.where(
    pricing_with_discount['expected_receiving_day'] < pd.Timestamp(TODAY),
    pd.NaT,
    pricing_with_discount['expected_receiving_day']
)
# Convert back to datetime (np.where returns object type)
pricing_with_discount['expected_receiving_day'] = pd.to_datetime(pricing_with_discount['expected_receiving_day'])

print(f"\nExpected receiving day calculated!")
print(f"Records with expected receiving day (future dates only): {len(pricing_with_discount[~pricing_with_discount['expected_receiving_day'].isna()])}")
print(f"Records with past expected dates (set to empty): {len(pricing_with_discount[pricing_with_discount['expected_receiving_day'].isna() & pricing_with_discount['last_po_date'].notna()])}")
print(f"Records with confirmation_status='no' (added 2 extra days): {len(pricing_with_discount[pricing_with_discount['confirmation_status'].str.lower() == 'no'])}")
print(f"\nSample data with expected receiving day:")
pricing_with_discount[~pricing_with_discount['last_po_date'].isna()][
    ['product_id', 'warehouse_id', 'sku', 'confirmation_status', 'last_po_date', 'leadtime', 'adjusted_leadtime', 'expected_receiving_day', 'doh']
].head(15)


Leadtime data added!

Records with leadtime: 104891
Records without leadtime: 0

Leadtime distribution:
count    104891.000000
mean         54.423583
std          30.318950
min          24.000000
25%          48.000000
50%          48.000000
75%          72.000000
max         168.000000
Name: leadtime, dtype: float64



Expected receiving day calculated!
Records with expected receiving day (future dates only): 11122
Records with past expected dates (set to empty): 11067
Records with confirmation_status='no' (added 2 extra days): 3516

Sample data with expected receiving day:


,product_id,warehouse_id,sku,confirmation_status,last_po_date,leadtime,adjusted_leadtime,expected_receiving_day,doh
0,4940,797,مناديل فاميليا مطبخ كلاسيك 6 رول + 2 رول هدية ...,no,2026-02-25,24.0,72.0,2026-02-28,0.000000
1,22318,632,اوريو بسكوت بكريمة بطعم الشوكولاتة والبندق - ...,yes,2026-02-23,24.0,24.0,NaT,8.000000
2,788,797,حلوانى مربى فراولة- 340 جم,yes,2026-02-23,48.0,48.0,NaT,10.000000
3,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,yes,2026-02-25,24.0,24.0,2026-02-26,5.823529
4,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,yes,2026-02-25,48.0,48.0,2026-02-27,5.823529
5,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,yes,2026-02-25,72.0,72.0,2026-02-28,5.823529
6,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,yes,2026-02-25,24.0,24.0,2026-02-26,5.823529
7,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,yes,2026-02-25,48.0,48.0,2026-02-27,5.823529
8,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,yes,2026-02-25,72.0,72.0,2026-02-28,5.823529
9,9786,236,عصير جهينة بيور جوافة كوكتيل - 235 مل,yes,2026-02-22,24.0,24.0,NaT,12.500000


In [33]:
# =============================================================================
# SKIP: Margin Boundaries Query - Now fetched via market_data_module.get_margin_tiers()
# =============================================================================
# The margin boundaries and tier calculation is now centralized in market_data_module.
# We'll use get_margin_tiers() to get pre-calculated margin tiers.

print("Loading margin tiers from market_data_module...")
df_margin_tiers = get_margin_tiers()
print(f"Loaded {len(df_margin_tiers)} margin tier records from module")


Loading margin tiers from market_data_module...

FETCHING MARGIN TIERS
Timestamp: 2026-02-26 08:07:10 Cairo time

Step 1: Fetching margin boundaries from PRODUCT_STATISTICS...


    Loaded 18684 records

Step 2: Adding cohort IDs...
    Records with cohorts: 25845

Step 3: Calculating margin tiers...

MARGIN TIERS COMPLETE
Total records: 25845

Margin Tier Structure:
  margin_tier_below:   effective_min - step (1 below)
  margin_tier_1:       effective_min_margin
  margin_tier_2:       effective_min + 1*step
  margin_tier_3:       effective_min + 2*step
  margin_tier_4:       effective_min + 3*step
  margin_tier_5:       max_boundary
  margin_tier_above_1: max_boundary + 1*step
  margin_tier_above_2: max_boundary + 2*step
Loaded 25845 margin tier records from module


In [34]:
# =============================================================================
# Add Margin Tiers from market_data_module (Pre-calculated)
# =============================================================================
# The margin tiers are now calculated in market_data_module.get_margin_tiers()
# We just need to merge them with pricing_with_discount

# Merge pre-calculated margin tiers
pricing_with_discount = pricing_with_discount.merge(
    df_margin_tiers[[
        'product_id', 'region', 'cohort_id',
        'optimal_bm', 'min_boundary', 'max_boundary', 'median_bm',
        'effective_min_margin', 'margin_step',
        'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
        'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2'
    ]], 
    on=['product_id', 'region','cohort_id'],
    how='left'
)

print(f"Margin tiers merged from module!")
print(f"\nRecords with margin tiers: {len(pricing_with_discount[~pricing_with_discount['max_boundary'].isna()])}")
print(f"Records without margin tiers: {len(pricing_with_discount[pricing_with_discount['max_boundary'].isna()])}")

print(f"\nMargin Tier Structure (from market_data_module):")
print(f"  margin_tier_below:   effective_min - step (1 below)")
print(f"  margin_tier_1:       effective_min_margin")
print(f"  margin_tier_2:       effective_min + 1*step")
print(f"  margin_tier_3:       effective_min + 2*step")
print(f"  margin_tier_4:       effective_min + 3*step")
print(f"  margin_tier_5:       max_boundary")
print(f"  margin_tier_above_1: max_boundary + 1*step")
print(f"  margin_tier_above_2: max_boundary + 2*step")

print(f"\nSample margin tiers:")
pricing_with_discount[~pricing_with_discount['max_boundary'].isna()][
    ['product_id', 'sku', 'effective_min_margin', 'max_boundary', 'margin_step',
     'margin_tier_below', 'margin_tier_1', 'margin_tier_3', 'margin_tier_5', 
     'margin_tier_above_1', 'margin_tier_above_2']
].head(10)


Margin tiers merged from module!

Records with margin tiers: 42845
Records without margin tiers: 62473

Margin Tier Structure (from market_data_module):
  margin_tier_below:   effective_min - step (1 below)
  margin_tier_1:       effective_min_margin
  margin_tier_2:       effective_min + 1*step
  margin_tier_3:       effective_min + 2*step
  margin_tier_4:       effective_min + 3*step
  margin_tier_5:       max_boundary
  margin_tier_above_1: max_boundary + 1*step
  margin_tier_above_2: max_boundary + 2*step

Sample margin tiers:


,product_id,sku,effective_min_margin,max_boundary,margin_step,margin_tier_below,margin_tier_1,margin_tier_3,margin_tier_5,margin_tier_above_1,margin_tier_above_2
0,4940,مناديل فاميليا مطبخ كلاسيك 6 رول + 2 رول هدية ...,0.044881,0.083005,0.009531,0.035349,0.044881,0.063943,0.083005,0.092536,0.102068
1,22318,اوريو بسكوت بكريمة بطعم الشوكولاتة والبندق - ...,0.045956,0.135855,0.022475,0.023481,0.045956,0.090905,0.135855,0.158330,0.180805
2,788,حلوانى مربى فراولة- 340 جم,0.012156,0.046432,0.008569,0.003587,0.012156,0.029294,0.046432,0.055001,0.063570
3,21683,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,0.061327,0.114179,0.013213,0.048114,0.061327,0.087753,0.114179,0.127392,0.140605
4,21683,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,0.061327,0.114179,0.013213,0.048114,0.061327,0.087753,0.114179,0.127392,0.140605
5,21683,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,0.061327,0.114179,0.013213,0.048114,0.061327,0.087753,0.114179,0.127392,0.140605
6,21683,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,0.061327,0.114179,0.013213,0.048114,0.061327,0.087753,0.114179,0.127392,0.140605
7,21683,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,0.061327,0.114179,0.013213,0.048114,0.061327,0.087753,0.114179,0.127392,0.140605
8,21683,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,0.061327,0.114179,0.013213,0.048114,0.061327,0.087753,0.114179,0.127392,0.140605
9,9786,عصير جهينة بيور جوافة كوكتيل - 235 مل,0.040423,0.066000,0.006394,0.034028,0.040423,0.053211,0.066000,0.072394,0.078789


In [35]:
# =============================================================================
# Minimum Selling Quantity Query - Get min selling qty per product
# =============================================================================
MIN_SELLING_QTY_QUERY = f'''
SELECT product_id, min_selling_qty
FROM (
    SELECT *, MIN(basic_unit_count) OVER (PARTITION BY product_id) AS min_selling_qty
    FROM (
        SELECT DISTINCT
            pso.product_id,
            pso.PACKING_UNIT_ID,
            pup.basic_unit_count,
            SUM(pso.total_price) AS nmv,
            SUM(pso.total_price) / SUM(nmv) OVER (PARTITION BY pso.product_id) AS cntrb
        FROM product_sales_order pso
        JOIN PACKING_UNIT_PRODUCTS pup ON pup.product_id = pso.product_id 
            AND pup.PACKING_UNIT_ID = pso.PACKING_UNIT_ID
        JOIN sales_orders so ON so.id = pso.sales_order_id
        WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
            AND so.sales_order_status_id NOT IN (7, 12)
            AND so.channel IN ('telesales', 'retailer')
            AND pso.purchased_item_count <> 0
        GROUP BY ALL
        QUALIFY cntrb > 0.05
    )
    QUALIFY basic_unit_count = min_selling_qty
)
'''

# Execute min selling qty query
print("Loading minimum selling quantity data...")
df_min_selling_qty = query_snowflake(MIN_SELLING_QTY_QUERY)
df_min_selling_qty = convert_to_numeric(df_min_selling_qty)
print(f"Loaded {len(df_min_selling_qty)} min selling qty records")


Loading minimum selling quantity data...


Loaded 3915 min selling qty records


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [36]:
# =============================================================================
# Add Min Selling Qty and Below Min Stock Flag to pricing_with_discount
# =============================================================================

# Merge min selling qty with pricing_with_discount (by product_id)
pricing_with_discount = pricing_with_discount.merge(
    df_min_selling_qty[['product_id', 'min_selling_qty']], 
    on='product_id', 
    how='left'
)

# Fill missing min_selling_qty with 1 (default)
pricing_with_discount['min_selling_qty'] = pricing_with_discount['min_selling_qty'].fillna(1).astype(int)

# Create flag: below_min_stock_flag = 1 if (RR = 0 AND stocks > 0 AND stocks < min_selling_qty)
pricing_with_discount['below_min_stock_flag'] = np.where(
    (pricing_with_discount['in_stock_rr'] == 0) & 
    (pricing_with_discount['stocks'] > 0) &
    (pricing_with_discount['stocks'] < pricing_with_discount['min_selling_qty']),
    1, 0
)

print(f"Min selling qty and below_min_stock_flag added!")
print(f"\nSKUs flagged (zero RR & stocks < min_selling_qty): {len(pricing_with_discount[pricing_with_discount['below_min_stock_flag'] == 1])}")
print(f"SKUs not flagged: {len(pricing_with_discount[pricing_with_discount['below_min_stock_flag'] == 0])}")
print(f"\nSample flagged SKUs:")
pricing_with_discount[pricing_with_discount['below_min_stock_flag'] == 1][
    ['product_id', 'warehouse_id', 'sku', 'stocks', 'min_selling_qty', 'in_stock_rr', 'below_min_stock_flag']
].head(15)


Min selling qty and below_min_stock_flag added!

SKUs flagged (zero RR & stocks < min_selling_qty): 15
SKUs not flagged: 105303

Sample flagged SKUs:


,product_id,warehouse_id,sku,stocks,min_selling_qty,in_stock_rr,below_min_stock_flag
14621,2213,401,اولويز الترا رفيعة طويل جدا 3*1 بالاعشاب - 7 فوطة,2,4,0.0,1
19629,71,632,عسل البوادى اسود - 355 جم,4,6,0.0,1
30811,12464,797,مولبد حماية ضد البكتيريا طويل جدا ماكسى مضغوطة...,1,4,0.0,1
34892,426,401,أمريكانا فول مدمس ساده- 400 جم,4,6,0.0,1
38126,426,797,أمريكانا فول مدمس ساده- 400 جم,3,6,0.0,1
48958,2213,797,اولويز الترا رفيعة طويل جدا 3*1 بالاعشاب - 7 فوطة,1,4,0.0,1
58948,10048,401,مولبد الترا رفيعة فاليو طويل جدا + 2 فوطة مجان...,1,2,0.0,1
60859,4674,797,التمساح عسل اسود - 680 جم,2,3,0.0,1
60860,4674,797,التمساح عسل اسود - 680 جم,2,3,0.0,1
63754,12464,1,مولبد حماية ضد البكتيريا طويل جدا ماكسى مضغوطة...,2,4,0.0,1


In [37]:
# =============================================================================
# Yesterday's Discount Analysis Query
# Gets: SKU discount, Quantity discount, Tier 1/2/3 NMV breakdown and contributions
# =============================================================================
YESTERDAY_DISCOUNT_QUERY = f'''
WITH qd_det AS (
    -- Map dynamic tags to warehouse IDs using name matching
    SELECT DISTINCT 
        dt.id AS tag_id, 
        dt.name AS tag_name,
        REPLACE(w.name, ' ', '') AS warehouse_name,
        w.id AS warehouse_id,
        warehouse_name ILIKE '%' || CASE 
            WHEN SPLIT_PART(tag_name, '_', 1) = 'El' THEN SPLIT_PART(tag_name, '_', 2) 
            ELSE SPLIT_PART(tag_name, '_', 1) 
        END || '%' AS contains_flag
    FROM dynamic_tags dt
    JOIN dynamic_taggables dta ON dt.id = dta.dynamic_tag_id 
    CROSS JOIN warehouses w 
    WHERE dt.id > 3000
        AND dt.name LIKE '%QD_rets%'
        AND w.id IN (1, 236, 337, 8, 339, 170, 501, 401, 703, 632, 797, 962)
        AND contains_flag = 'true'
),

qd_config AS (
    SELECT * 
    FROM (
        SELECT 
            product_id,
            start_at,
            end_at,
            packing_unit_id,
            id AS qd_id,
            qd.warehouse_id,
            MAX(CASE WHEN tier = 1 THEN quantity END) AS tier_1_qty,
            MAX(CASE WHEN tier = 1 THEN discount_percentage END) AS tier_1_discount_pct,
            MAX(CASE WHEN tier = 2 THEN quantity END) AS tier_2_qty,
            MAX(CASE WHEN tier = 2 THEN discount_percentage END) AS tier_2_discount_pct,
            MAX(CASE WHEN tier = 3 THEN quantity END) AS tier_3_qty,
            MAX(CASE WHEN tier = 3 THEN discount_percentage END) AS tier_3_discount_pct
        FROM (
            SELECT 
                qd.id,
                qdv.product_id,
                qdv.packing_unit_id,
                qdv.quantity,
                qdv.discount_percentage,
                qd.dynamic_tag_id,
                qd.start_at,
                qd.end_at,
                ROW_NUMBER() OVER (
                    PARTITION BY qdv.product_id, qdv.packing_unit_id, qd.id 
                    ORDER BY qdv.quantity
                ) AS tier
            FROM quantity_discounts qd 
            JOIN quantity_discount_values qdv ON qd.id = qdv.quantity_discount_id 
            WHERE CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1 
                  BETWEEN qd.start_at::DATE AND qd.end_at::DATE
                AND qd.start_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 5
        ) qd_tiers
        JOIN qd_det qd ON qd.tag_id = qd_tiers.dynamic_tag_id
        GROUP BY ALL
    )
    QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id, packing_unit_id, warehouse_id ORDER BY start_at DESC) = 1
),

-- Get all sales from yesterday
yesterday_sales AS (
    SELECT 
        pso.warehouse_id,
        pso.product_id,
        so.retailer_id,
        pso.packing_unit_id,
        pso.purchased_item_count AS qty,
        pso.total_price AS nmv,
        pso.item_price / pso.basic_unit_count AS unit_price,
        pso.ITEM_DISCOUNT_VALUE AS sku_discount_per_unit,
        pso.ITEM_QUANTITY_DISCOUNT_VALUE AS qty_discount_per_unit,
        pso.ITEM_DISCOUNT_VALUE * pso.purchased_item_count AS sku_discount_total,
        pso.ITEM_QUANTITY_DISCOUNT_VALUE * pso.purchased_item_count AS qty_discount_total,
        qd.tier_1_qty,
        qd.tier_2_qty,
        qd.tier_3_qty,
        qd.tier_1_discount_pct,
        qd.tier_2_discount_pct,
        qd.tier_3_discount_pct,
        -- Determine tier used
        CASE 
            WHEN pso.ITEM_QUANTITY_DISCOUNT_VALUE = 0 OR qd.tier_1_qty IS NULL THEN 'Base'
            WHEN qd.tier_3_qty IS NOT NULL AND pso.purchased_item_count >= qd.tier_3_qty THEN 'Tier 3'
            WHEN qd.tier_2_qty IS NOT NULL AND pso.purchased_item_count >= qd.tier_2_qty THEN 'Tier 2'
            WHEN qd.tier_1_qty IS NOT NULL AND pso.purchased_item_count >= qd.tier_1_qty THEN 'Tier 1'
            ELSE 'Base'
        END AS tier_used
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    LEFT JOIN qd_config qd 
        ON qd.product_id = pso.product_id 
        AND qd.packing_unit_id = pso.packing_unit_id
        AND qd.warehouse_id = so.warehouse_id
    WHERE so.created_at::DATE = CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
)

SELECT 
    warehouse_id,
    product_id,
    SUM(nmv) AS total_nmv,
    SUM(CASE WHEN sku_discount_per_unit > 0 THEN nmv ELSE 0 END) AS sku_discount_nmv,
    SUM(CASE WHEN qty_discount_per_unit > 0 THEN nmv ELSE 0 END) AS qty_discount_nmv,
    SUM(CASE WHEN tier_used = 'Tier 1' THEN nmv ELSE 0 END) AS tier1_nmv,
    SUM(CASE WHEN tier_used = 'Tier 2' THEN nmv ELSE 0 END) AS tier2_nmv,
    SUM(CASE WHEN tier_used = 'Tier 3' THEN nmv ELSE 0 END) AS tier3_nmv,
    -- Tier quantities and discount percentages (from the active QD config)
    MAX(tier_1_qty) AS tier_1_qty,
    MAX(tier_1_discount_pct) AS tier_1_discount_pct,
    MAX(tier_2_qty) AS tier_2_qty,
    MAX(tier_2_discount_pct) AS tier_2_discount_pct,
    MAX(tier_3_qty) AS tier_3_qty,
    MAX(tier_3_discount_pct) AS tier_3_discount_pct
FROM yesterday_sales
GROUP BY warehouse_id, product_id
HAVING SUM(nmv) > 0
ORDER BY total_nmv DESC
'''

# Execute yesterday discount query
print("Loading yesterday's discount analysis data...")
df_yesterday_discount = query_snowflake(YESTERDAY_DISCOUNT_QUERY)
df_yesterday_discount = convert_to_numeric(df_yesterday_discount)
print(f"Loaded {len(df_yesterday_discount)} SKU discount records from yesterday")

# Calculate contributions in Python
df_yesterday_discount['sku_discount_nmv_cntrb'] = (
    df_yesterday_discount['sku_discount_nmv'] / df_yesterday_discount['total_nmv'] * 100
).round(2)
df_yesterday_discount['qty_discount_nmv_cntrb'] = (
    df_yesterday_discount['qty_discount_nmv'] / df_yesterday_discount['total_nmv'] * 100
).round(2)
df_yesterday_discount['tier1_nmv_cntrb'] = (
    df_yesterday_discount['tier1_nmv'] / df_yesterday_discount['total_nmv'] * 100
).round(2)
df_yesterday_discount['tier2_nmv_cntrb'] = (
    df_yesterday_discount['tier2_nmv'] / df_yesterday_discount['total_nmv'] * 100
).round(2)
df_yesterday_discount['tier3_nmv_cntrb'] = (
    df_yesterday_discount['tier3_nmv'] / df_yesterday_discount['total_nmv'] * 100
).round(2)

# Summary
print(f"\n{'='*60}")
print(f"YESTERDAY'S DISCOUNT ANALYSIS SUMMARY")
print(f"{'='*60}")
print(f"\nTotal NMV yesterday: {df_yesterday_discount['total_nmv'].sum():,.0f}")
print(f"SKU Discount NMV: {df_yesterday_discount['sku_discount_nmv'].sum():,.0f}")
print(f"Quantity Discount NMV: {df_yesterday_discount['qty_discount_nmv'].sum():,.0f}")
print(f"\nNMV by Tier:")
print(f"  Tier 1: {df_yesterday_discount['tier1_nmv'].sum():,.0f}")
print(f"  Tier 2: {df_yesterday_discount['tier2_nmv'].sum():,.0f}")
print(f"  Tier 3: {df_yesterday_discount['tier3_nmv'].sum():,.0f}")

df_yesterday_discount.head(10)


Loading yesterday's discount analysis data...


Loaded 11169 SKU discount records from yesterday

YESTERDAY'S DISCOUNT ANALYSIS SUMMARY

Total NMV yesterday: 22,145,921
SKU Discount NMV: 6,339,826
Quantity Discount NMV: 2,533,026

NMV by Tier:
  Tier 1: 746,738
  Tier 2: 1,228,164
  Tier 3: 490,346


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,warehouse_id,product_id,total_nmv,sku_discount_nmv,qty_discount_nmv,tier1_nmv,tier2_nmv,tier3_nmv,tier_1_qty,tier_1_discount_pct,tier_2_qty,tier_2_discount_pct,tier_3_qty,tier_3_discount_pct,sku_discount_nmv_cntrb,qty_discount_nmv_cntrb,tier1_nmv_cntrb,tier2_nmv_cntrb,tier3_nmv_cntrb
0,1,589,75367.940,42653.75,0.00,0.0,0.0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,56.59,0.00,0.00,0.00,0.00
1,170,434,69136.000,50225.50,45507.00,7092.0,38415.0,0.00,4.0,0.50,7.0,1.00,NaN,NaN,72.65,65.82,10.26,55.56,0.00
2,501,2778,68814.500,27832.00,0.00,0.0,0.0,0.00,5.0,1.04,13.0,2.96,NaN,NaN,40.44,0.00,0.00,0.00,0.00
3,1,6935,59116.500,45341.50,0.00,0.0,0.0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,76.70,0.00,0.00,0.00,0.00
4,1,151,58793.000,14994.00,7938.00,0.0,7938.0,0.00,4.0,0.85,7.0,1.64,73.0,2.51,25.50,13.50,0.00,13.50,0.00
5,1,3,56361.365,7945.25,26962.75,985.0,6524.0,19453.75,4.0,0.50,7.0,1.00,66.0,1.68,14.10,47.84,1.75,11.58,34.52
6,1,990,55538.000,14844.00,3850.00,0.0,3850.0,0.00,4.0,0.67,7.0,1.11,30.0,1.68,26.73,6.93,0.00,6.93,0.00
7,236,130,53636.000,14316.75,20131.00,0.0,3437.0,16694.00,4.0,0.50,7.0,1.00,34.0,1.54,26.69,37.53,0.00,6.41,31.12
8,1,130,50436.515,15498.00,1994.00,1994.0,0.0,0.00,4.0,0.40,7.0,0.91,33.0,1.68,30.73,3.95,3.95,0.00,0.00
9,501,1504,49519.500,7311.00,0.00,0.0,0.0,0.00,5.0,1.28,13.0,3.66,NaN,NaN,14.76,0.00,0.00,0.00,0.00


In [38]:
# =============================================================================
# Add Yesterday's Discount Analysis to pricing_with_discount (Contributions Only)
# =============================================================================

# Merge yesterday discount data with pricing_with_discount - contributions + tier config
pricing_with_discount = pricing_with_discount.merge(
    df_yesterday_discount[[
        'warehouse_id', 'product_id', 
        'sku_discount_nmv_cntrb', 'qty_discount_nmv_cntrb',
        'tier1_nmv_cntrb', 'tier2_nmv_cntrb', 'tier3_nmv_cntrb',
        'tier_1_qty', 'tier_1_discount_pct',
        'tier_2_qty', 'tier_2_discount_pct',
        'tier_3_qty', 'tier_3_discount_pct'
    ]].rename(columns={
        'sku_discount_nmv_cntrb': 'yesterday_sku_disc_cntrb',
        'qty_discount_nmv_cntrb': 'yesterday_qty_disc_cntrb',
        'tier1_nmv_cntrb': 'yesterday_t1_cntrb',
        'tier2_nmv_cntrb': 'yesterday_t2_cntrb',
        'tier3_nmv_cntrb': 'yesterday_t3_cntrb',
        'tier_1_qty': 'qd_tier_1_qty',
        'tier_1_discount_pct': 'qd_tier_1_disc_pct',
        'tier_2_qty': 'qd_tier_2_qty',
        'tier_2_discount_pct': 'qd_tier_2_disc_pct',
        'tier_3_qty': 'qd_tier_3_qty',
        'tier_3_discount_pct': 'qd_tier_3_disc_pct'
    }), 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill NaN for SKUs that had no sales yesterday
contrib_cols = [
    'yesterday_sku_disc_cntrb', 'yesterday_qty_disc_cntrb',
    'yesterday_t1_cntrb', 'yesterday_t2_cntrb', 'yesterday_t3_cntrb'
]
for col in contrib_cols:
    if col in pricing_with_discount.columns:
        pricing_with_discount[col] = pricing_with_discount[col].fillna(0)

# Fill NaN for QD tier config (0 means no tier configured)
qd_config_cols = [
    'qd_tier_1_qty', 'qd_tier_1_disc_pct',
    'qd_tier_2_qty', 'qd_tier_2_disc_pct',
    'qd_tier_3_qty', 'qd_tier_3_disc_pct'
]
for col in qd_config_cols:
    if col in pricing_with_discount.columns:
        pricing_with_discount[col] = pricing_with_discount[col].fillna(0)

print(f"Yesterday's discount contributions and QD tier config added!")
print(f"\nSKUs with discount data: {len(pricing_with_discount[pricing_with_discount['yesterday_sku_disc_cntrb'] > 0]) + len(pricing_with_discount[pricing_with_discount['yesterday_qty_disc_cntrb'] > 0])}")
print(f"SKUs with QD tier config: {len(pricing_with_discount[pricing_with_discount['qd_tier_1_qty'] > 0])}")
print(f"\nSample data with yesterday's discount contributions and QD tiers:")
pricing_with_discount[pricing_with_discount['qd_tier_1_qty'] > 0][
    ['product_id', 'warehouse_id', 'sku', 
     'yesterday_sku_disc_cntrb', 'yesterday_qty_disc_cntrb',
     'qd_tier_1_qty', 'qd_tier_1_disc_pct', 'qd_tier_2_qty', 'qd_tier_2_disc_pct', 'qd_tier_3_qty', 'qd_tier_3_disc_pct']
].head(15)


Yesterday's discount contributions and QD tier config added!

SKUs with discount data: 5618
SKUs with QD tier config: 2851

Sample data with yesterday's discount contributions and QD tiers:


,product_id,warehouse_id,sku,yesterday_sku_disc_cntrb,yesterday_qty_disc_cntrb,qd_tier_1_qty,qd_tier_1_disc_pct,qd_tier_2_qty,qd_tier_2_disc_pct,qd_tier_3_qty,qd_tier_3_disc_pct
28,8635,236,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,64.25,35.82,4.0,0.83,7.0,1.61,455.0,2.37
29,8635,236,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,64.25,35.82,4.0,0.83,7.0,1.61,455.0,2.37
32,950,401,لبن جهينة كامل الدسم - 1.5 لتر,0.00,0.00,4.0,0.64,7.0,1.24,27.0,1.94
42,151,236,لبن بخيره - 1 لتر,0.00,0.00,4.0,0.50,7.0,1.00,38.0,1.54
43,151,962,لبن بخيره - 1 لتر,0.00,24.43,4.0,0.50,7.0,1.00,38.0,1.53
52,62,962,نسكافيه كلاسيك - 1.8 جم,0.57,31.14,8.0,0.50,14.0,1.00,204.0,1.53
53,62,962,نسكافيه كلاسيك - 1.8 جم,0.57,31.14,8.0,0.50,14.0,1.00,204.0,1.53
56,205,632,لبن جهينة مكس شوكولاتة - 200 مل,71.43,0.00,4.0,0.53,7.0,1.00,51.0,1.94
58,206,797,لبن جهينة مكس فراولة - 200 مل,50.36,0.00,4.0,0.50,7.0,1.00,45.0,1.89
62,2878,1,زيت كريستال عباد الشمس 6 زجاجة - 700 مل,50.21,0.00,4.0,0.50,7.0,1.00,44.0,1.68


In [39]:
# =============================================================================
# Performance Benchmark Query
# Gets: Yesterday qty, Recent 7d qty, MTD qty, and P80 benchmarks (240 days)
# Uses materialized_views.stock_day_close for in-stock determination
# =============================================================================
PERFORMANCE_BENCHMARK_QUERY = f'''
-- =============================================================================
-- PERFORMANCE BENCHMARK QUERY WITH FALLBACK LOGIC
-- =============================================================================
-- This query provides performance benchmarks (P80) for SKUs with fallback logic:
-- 1. Primary: p80_daily_240d (calculated from 240-day history)
-- 2. Fallback 1: Median cat-brand quantity (per warehouse)
-- 3. Fallback 2: Median cat quantity (per warehouse)
-- 4. Final fallback: 5
-- =============================================================================

WITH params AS (
    SELECT
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE AS today,
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1 AS yesterday,
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 240 AS history_start,
        DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) AS current_month_start,
        DAY(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) AS current_day_of_month
),

-- Product category and brand lookup
product_lookup AS (
    SELECT DISTINCT
        p.id AS product_id,
        b.name_ar AS brand,
        cat.name_ar AS cat
    FROM products p
    JOIN brands b ON b.id = p.brand_id
    JOIN categories cat ON cat.id = p.category_id
),

-- Daily sales aggregation (240 days) - includes qty and retailer count
daily_sales AS (
    SELECT
        pso.warehouse_id,
        pso.product_id,
        so.created_at::DATE AS sale_date,
        SUM(pso.purchased_item_count * pso.basic_unit_count) AS daily_qty,
        COUNT(DISTINCT so.retailer_id) AS daily_retailers
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    CROSS JOIN params p
    WHERE so.created_at::DATE >= p.history_start
        AND so.created_at::DATE < p.today
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY pso.warehouse_id, pso.product_id, so.created_at::DATE
),

-- Daily stock status using stock_day_close
-- In-stock = opening (prev day close) > 0 AND closing > 0
daily_stock AS (
    SELECT
        sdc.warehouse_id,
        sdc.product_id,
        sdc.TIMESTAMP::DATE AS stock_date,
        sdc.available_stock,
        LAG(sdc.available_stock, 1) OVER (
            PARTITION BY sdc.warehouse_id, sdc.product_id 
            ORDER BY sdc.TIMESTAMP::DATE
        ) AS opening_stock,
        CASE 
            WHEN LAG(sdc.available_stock, 1) OVER (
                    PARTITION BY sdc.warehouse_id, sdc.product_id ORDER BY sdc.TIMESTAMP::DATE
                 ) > 0 
                 AND sdc.available_stock > 0 
            THEN 1 
            ELSE 0 
        END AS in_stock_flag
    FROM materialized_views.stock_day_close sdc
    CROSS JOIN params p
    WHERE sdc.TIMESTAMP::DATE >= p.history_start - 1  -- Need one extra day for LAG
        AND sdc.TIMESTAMP::DATE < p.today
),

-- Combine sales with stock status
daily_with_stock AS (
    SELECT
        COALESCE(ds.warehouse_id, st.warehouse_id) AS warehouse_id,
        COALESCE(ds.product_id, st.product_id) AS product_id,
        COALESCE(ds.sale_date, st.stock_date) AS the_date,
        COALESCE(ds.daily_qty, 0) AS daily_qty,
        COALESCE(ds.daily_retailers, 0) AS daily_retailers,
        COALESCE(st.in_stock_flag, 0) AS in_stock_flag
    FROM daily_sales ds
    FULL OUTER JOIN daily_stock st 
        ON ds.warehouse_id = st.warehouse_id 
        AND ds.product_id = st.product_id 
        AND ds.sale_date = st.stock_date
    WHERE COALESCE(ds.sale_date, st.stock_date) >= (SELECT history_start FROM params)
),

-- Add product category and brand to daily_with_stock for fallback calculations
daily_with_stock_cat_brand AS (
    SELECT
        dws.*,
        pl.brand,
        pl.cat
    FROM daily_with_stock dws
    LEFT JOIN product_lookup pl ON pl.product_id = dws.product_id
),

-- Identify new SKUs: those with sales ONLY in current month (no historical sales before current month)
new_sku_identification AS (
    SELECT
        warehouse_id,
        product_id,
        CASE 
            WHEN MAX(CASE WHEN the_date < (SELECT current_month_start FROM params) THEN 1 ELSE 0 END) = 0
                 AND MAX(CASE WHEN the_date >= (SELECT current_month_start FROM params) AND the_date < (SELECT today FROM params) THEN 1 ELSE 0 END) = 1
            THEN 1 
            ELSE 0 
        END AS is_new_sku
    FROM daily_with_stock
    WHERE daily_qty > 0  -- Only consider days with actual sales
    GROUP BY warehouse_id, product_id
),

-- Calculate P80 benchmark (in-stock days only, 240 days, EXCLUDING last 7 days)
p80_daily_benchmark AS (
    SELECT
        warehouse_id,
        product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY daily_qty) AS p80_daily_240d,
        AVG(daily_qty) AS avg_daily_240d,
        STDDEV(daily_qty) AS std_daily_240d,
        COUNT(*) AS in_stock_days_240d
    FROM daily_with_stock
    CROSS JOIN params p
    WHERE in_stock_flag = 1
        AND the_date >= p.history_start
        AND the_date < p.today - 7  -- Exclude last 7 days from benchmark
    GROUP BY warehouse_id, product_id
),

-- Calculate median cat-brand quantity (fallback 1)
-- Same filters: 240 days, in-stock only, exclude last 7 days, per warehouse
cat_brand_median AS (
    SELECT
        dws.warehouse_id,
        dws.cat,
        dws.brand,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dws.daily_qty) AS median_cat_brand_qty
    FROM daily_with_stock_cat_brand dws
    CROSS JOIN params p
    WHERE dws.in_stock_flag = 1
        AND dws.the_date >= p.history_start
        AND dws.the_date < p.today - 7  -- Exclude last 7 days from benchmark
        AND dws.cat IS NOT NULL
        AND dws.brand IS NOT NULL
    GROUP BY dws.warehouse_id, dws.cat, dws.brand
),

-- Calculate median cat quantity (fallback 2)
-- Same filters: 240 days, in-stock only, exclude last 7 days, per warehouse
cat_median AS (
    SELECT
        dws.warehouse_id,
        dws.cat,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dws.daily_qty) AS median_cat_qty
    FROM daily_with_stock_cat_brand dws
    CROSS JOIN params p
    WHERE dws.in_stock_flag = 1
        AND dws.the_date >= p.history_start
        AND dws.the_date < p.today - 7  -- Exclude last 7 days from benchmark
        AND dws.cat IS NOT NULL
    GROUP BY dws.warehouse_id, dws.cat
),

-- Calculate P70 retailer benchmark (in-stock days only, 240 days, EXCLUDING last 7 days)
p70_retailer_benchmark AS (
    SELECT
        warehouse_id,
        product_id,
        PERCENTILE_CONT(0.7) WITHIN GROUP (ORDER BY daily_retailers) AS p70_daily_retailers_240d,
        AVG(daily_retailers) AS avg_daily_retailers_240d,
        STDDEV(daily_retailers) AS std_daily_retailers_240d
    FROM daily_with_stock
    CROSS JOIN params p
    WHERE in_stock_flag = 1
        AND the_date >= p.history_start
        AND the_date < p.today - 7  -- Exclude last 7 days from benchmark
    GROUP BY warehouse_id, product_id
),

-- NEW: Calculate current-month-only benchmarks for new SKUs
-- Use previous days of current month (excluding yesterday and last 1-2 days for stability)
current_month_benchmark AS (
    SELECT
        dws.warehouse_id,
        dws.product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY dws.daily_qty) AS p80_daily_current_month,
        AVG(dws.daily_qty) AS avg_daily_current_month,
        STDDEV(dws.daily_qty) AS std_daily_current_month,
        COUNT(*) AS in_stock_days_current_month
    FROM daily_with_stock dws
    CROSS JOIN params p
    INNER JOIN new_sku_identification nsi 
        ON dws.warehouse_id = nsi.warehouse_id 
        AND dws.product_id = nsi.product_id
    WHERE dws.in_stock_flag = 1
        AND dws.the_date >= p.current_month_start
        AND dws.the_date < p.yesterday - 1  -- Exclude yesterday and day before for stability
        AND nsi.is_new_sku = 1
    GROUP BY dws.warehouse_id, dws.product_id
    HAVING COUNT(*) >= 2  -- Need at least 2 days of data
),

-- NEW: Calculate current-month retailer benchmark for new SKUs
current_month_retailer_benchmark AS (
    SELECT
        dws.warehouse_id,
        dws.product_id,
        PERCENTILE_CONT(0.7) WITHIN GROUP (ORDER BY dws.daily_retailers) AS p70_daily_retailers_current_month,
        AVG(dws.daily_retailers) AS avg_daily_retailers_current_month,
        STDDEV(dws.daily_retailers) AS std_daily_retailers_current_month
    FROM daily_with_stock dws
    CROSS JOIN params p
    INNER JOIN new_sku_identification nsi 
        ON dws.warehouse_id = nsi.warehouse_id 
        AND dws.product_id = nsi.product_id
    WHERE dws.in_stock_flag = 1
        AND dws.the_date >= p.current_month_start
        AND dws.the_date < p.yesterday - 1  -- Exclude yesterday and day before
        AND nsi.is_new_sku = 1
    GROUP BY dws.warehouse_id, dws.product_id
    HAVING COUNT(*) >= 2
),

-- Calculate 7-day rolling SUM for P80 recent benchmark
rolling_7d AS (
    SELECT
        warehouse_id,
        product_id,
        the_date,
        SUM(daily_qty) OVER (
            PARTITION BY warehouse_id, product_id 
            ORDER BY the_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS rolling_7d_sum,
        SUM(in_stock_flag) OVER (
            PARTITION BY warehouse_id, product_id 
            ORDER BY the_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS in_stock_days_7d
    FROM daily_with_stock
),

p80_7d_benchmark AS (
    SELECT
        warehouse_id,
        product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY rolling_7d_sum) AS p80_7d_rolling_240d
    FROM rolling_7d
    CROSS JOIN params p
    WHERE the_date >= p.history_start + 7  -- Need 7 days for rolling
        AND the_date < p.today - 7  -- Exclude last 7 days from benchmark
        AND in_stock_days_7d >= 4  -- At least 4 of 7 days in stock
    GROUP BY warehouse_id, product_id
),

-- NEW: Calculate 7-day rolling benchmark for new SKUs using current month data
current_month_7d_benchmark AS (
    SELECT
        r7d.warehouse_id,
        r7d.product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY r7d.rolling_7d_sum) AS p80_7d_rolling_current_month
    FROM rolling_7d r7d
    CROSS JOIN params p
    INNER JOIN new_sku_identification nsi 
        ON r7d.warehouse_id = nsi.warehouse_id 
        AND r7d.product_id = nsi.product_id
    WHERE r7d.the_date >= p.current_month_start + 7  -- Need 7 days for rolling
        AND r7d.the_date < p.yesterday - 1  -- Exclude last 1-2 days
        AND r7d.in_stock_days_7d >= 4  -- At least 4 of 7 days in stock
        AND nsi.is_new_sku = 1
    GROUP BY r7d.warehouse_id, r7d.product_id
),

-- MTD benchmark: P80 of same MTD period totals (last 12 months)
-- Sum all sales from day 1 to current day of month for each historical month
mtd_historical AS (
    SELECT
        dws.warehouse_id,
        dws.product_id,
        DATE_TRUNC('month', dws.the_date) AS period_month_start,
        SUM(dws.daily_qty) AS mtd_total_qty  -- Sum of all days from 1 to current_day_of_month
    FROM daily_with_stock dws
    CROSS JOIN params p
    WHERE DAY(dws.the_date) <= p.current_day_of_month  -- Only days up to current day of month
    GROUP BY dws.warehouse_id, dws.product_id, DATE_TRUNC('month', dws.the_date)
),

mtd_by_period AS (
    SELECT
        mh.warehouse_id,
        mh.product_id,
        mh.period_month_start,
        mh.mtd_total_qty AS mtd_qty_at_day  -- Total MTD qty for that month
    FROM mtd_historical mh
    CROSS JOIN params p
    WHERE mh.period_month_start >= DATEADD(month, -12, p.current_month_start)
        AND mh.period_month_start < p.current_month_start
),

p80_mtd_benchmark AS (
    SELECT
        warehouse_id,
        product_id,
        PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY mtd_qty_at_day) AS p80_mtd_12mo,
        AVG(mtd_qty_at_day) AS avg_mtd_12mo
    FROM mtd_by_period
    GROUP BY warehouse_id, product_id
    HAVING COUNT(*) >= 3  -- At least 3 months of data
),

-- Current period quantities
current_metrics AS (
    SELECT
        warehouse_id,
        product_id,
        -- Yesterday
        SUM(CASE WHEN the_date = (SELECT yesterday FROM params) THEN daily_qty ELSE 0 END) AS yesterday_qty,
        SUM(CASE WHEN the_date = (SELECT yesterday FROM params) THEN daily_retailers ELSE 0 END) AS yesterday_retailers,
        -- Yesterday in-stock flag (1 if in stock yesterday, 0 otherwise)
        MAX(CASE WHEN the_date = (SELECT yesterday FROM params) THEN in_stock_flag ELSE 0 END) AS yesterday_in_stock,
        -- Recent 7 days
        SUM(CASE WHEN the_date >= (SELECT today FROM params) - 7 AND the_date < (SELECT today FROM params) THEN daily_qty ELSE 0 END) AS recent_7d_qty,
        SUM(CASE WHEN the_date >= (SELECT today FROM params) - 7 AND the_date < (SELECT today FROM params) AND in_stock_flag = 1 THEN 1 ELSE 0 END) AS recent_7d_in_stock_days,
        -- MTD
        SUM(CASE WHEN the_date >= (SELECT current_month_start FROM params) AND the_date < (SELECT today FROM params) THEN daily_qty ELSE 0 END) AS mtd_qty,
        SUM(CASE WHEN the_date >= (SELECT current_month_start FROM params) AND the_date < (SELECT today FROM params) AND in_stock_flag = 1 THEN 1 ELSE 0 END) AS mtd_in_stock_days
    FROM daily_with_stock
    GROUP BY warehouse_id, product_id
),

-- Combined performance ratio calculation with dynamic weights and fallback logic
-- Priority: New SKU current month benchmarks > Historical benchmarks > Fallback (cat-brand/cat median/5)
combined_ratio_calc AS (
    SELECT
        cm.warehouse_id,
        cm.product_id,
        cm.yesterday_qty,
        cm.yesterday_retailers,
        cm.yesterday_in_stock,
        cm.recent_7d_qty,
        cm.recent_7d_in_stock_days,
        cm.mtd_qty,
        cm.mtd_in_stock_days,
        
        -- Check if this is a new SKU
        COALESCE(nsi.is_new_sku, 0) AS is_new_sku,
        
        -- Get product category and brand for fallback lookup
        pl.brand,
        pl.cat,
        
        -- Benchmark values: For new SKUs, use current month benchmarks first, then fallback
        -- For existing SKUs, use historical benchmarks with fallback
        COALESCE(
            -- If new SKU and has current month benchmark, use it
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
            -- Otherwise use historical benchmark
            pb.p80_daily_240d,
            -- Fallback chain
            cbm.median_cat_brand_qty,
            cm_median.median_cat_qty,
            5
        ) AS p80_daily_240d,
        
        -- Average, stddev, and in_stock_days: Use current month for new SKUs, historical for others
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.avg_daily_current_month ELSE NULL END,
            pb.avg_daily_240d,
            0
        ) AS avg_daily_240d,
        
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.std_daily_current_month ELSE NULL END,
            pb.std_daily_240d,
            0
        ) AS std_daily_240d,
        
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.in_stock_days_current_month ELSE NULL END,
            pb.in_stock_days_240d,
            0
        ) AS in_stock_days_240d,
        
        -- For 7d: Use current month for new SKUs, historical for others, then fallback
        COALESCE(
            -- If new SKU and has current month 7d benchmark, use it
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cm7d.p80_7d_rolling_current_month ELSE NULL END,
            -- Otherwise use historical 7d benchmark
            p7.p80_7d_rolling_240d,
            -- Fallback: use daily benchmark * 7
            COALESCE(
                CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
                pb.p80_daily_240d,
                cbm.median_cat_brand_qty,
                cm_median.median_cat_qty,
                5
            ) * 7,
            35  -- 5 * 7 as final fallback
        ) AS p80_7d_sum_240d,
        
        -- For MTD: Use current month daily * days for new SKUs, historical MTD for others, then fallback
        COALESCE(
            -- If new SKU, use current month daily benchmark * days
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 
                 THEN cmb.p80_daily_current_month * (SELECT current_day_of_month FROM params)
                 ELSE NULL 
            END,
            -- Otherwise use historical MTD benchmark
            pm.p80_mtd_12mo,
            -- Fallback: use daily benchmark * days
            COALESCE(
                CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
                pb.p80_daily_240d,
                cbm.median_cat_brand_qty,
                cm_median.median_cat_qty,
                5
            ) * (SELECT current_day_of_month FROM params),
            5 * (SELECT current_day_of_month FROM params)  -- Final fallback
        ) AS p80_mtd_12mo,
        
        -- Retailer benchmarks: Use current month for new SKUs, historical for others
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmrb.p70_daily_retailers_current_month ELSE NULL END,
            pr.p70_daily_retailers_240d,
            1
        ) AS p70_daily_retailers_240d,
        
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmrb.avg_daily_retailers_current_month ELSE NULL END,
            pr.avg_daily_retailers_240d,
            0
        ) AS avg_daily_retailers_240d,
        
        COALESCE(
            CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmrb.std_daily_retailers_current_month ELSE NULL END,
            pr.std_daily_retailers_240d,
            0
        ) AS std_daily_retailers_240d,
        
        -- Calculate base ratios (capped at 3) using benchmarks with new SKU priority
        LEAST(
            cm.yesterday_qty / NULLIF(
                COALESCE(
                    CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
                    pb.p80_daily_240d,
                    cbm.median_cat_brand_qty,
                    cm_median.median_cat_qty,
                    5
                ), 0
            ), 3
        ) AS yesterday_ratio_capped,
        
        LEAST(
            cm.recent_7d_qty / NULLIF(
                COALESCE(
                    CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cm7d.p80_7d_rolling_current_month ELSE NULL END,
                    p7.p80_7d_rolling_240d,
                    COALESCE(
                        CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
                        pb.p80_daily_240d,
                        cbm.median_cat_brand_qty,
                        cm_median.median_cat_qty,
                        5
                    ) * 7,
                    35
                ), 0
            ), 3
        ) AS recent_ratio_capped,
        
        LEAST(
            cm.mtd_qty / NULLIF(
                COALESCE(
                    CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 
                         THEN cmb.p80_daily_current_month * (SELECT current_day_of_month FROM params)
                         ELSE NULL 
                    END,
                    pm.p80_mtd_12mo,
                    COALESCE(
                        CASE WHEN COALESCE(nsi.is_new_sku, 0) = 1 THEN cmb.p80_daily_current_month ELSE NULL END,
                        pb.p80_daily_240d,
                        cbm.median_cat_brand_qty,
                        cm_median.median_cat_qty,
                        5
                    ) * (SELECT current_day_of_month FROM params),
                    5 * (SELECT current_day_of_month FROM params)
                ), 0
            ), 3
        ) AS mtd_ratio_capped,
        
        -- In-stock percentages for each period
        cm.yesterday_in_stock AS yesterday_in_stock_pct,
        cm.recent_7d_in_stock_days / 7.0 AS recent_7d_in_stock_pct,
        cm.mtd_in_stock_days / NULLIF((SELECT current_day_of_month FROM params) - 1, 0) AS mtd_in_stock_pct,
        
        -- Raw weights (base: 20% yesterday, 40% recent 7d, 40% MTD) scaled by in-stock percentage
        0.2 * cm.yesterday_in_stock AS yesterday_raw_weight,
        0.4 * (cm.recent_7d_in_stock_days / 7.0) AS recent_7d_raw_weight,
        CASE 
            WHEN (SELECT current_day_of_month FROM params) >= 3 
            THEN 0.4 * COALESCE(cm.mtd_in_stock_days / NULLIF((SELECT current_day_of_month FROM params) - 1, 0), 0)
            ELSE 0 
        END AS mtd_raw_weight
        
    FROM current_metrics cm
    LEFT JOIN new_sku_identification nsi ON cm.warehouse_id = nsi.warehouse_id AND cm.product_id = nsi.product_id
    LEFT JOIN p80_daily_benchmark pb ON cm.warehouse_id = pb.warehouse_id AND cm.product_id = pb.product_id
    LEFT JOIN current_month_benchmark cmb ON cm.warehouse_id = cmb.warehouse_id AND cm.product_id = cmb.product_id
    LEFT JOIN p80_7d_benchmark p7 ON cm.warehouse_id = p7.warehouse_id AND cm.product_id = p7.product_id
    LEFT JOIN current_month_7d_benchmark cm7d ON cm.warehouse_id = cm7d.warehouse_id AND cm.product_id = cm7d.product_id
    LEFT JOIN p80_mtd_benchmark pm ON cm.warehouse_id = pm.warehouse_id AND cm.product_id = pm.product_id
    LEFT JOIN p70_retailer_benchmark pr ON cm.warehouse_id = pr.warehouse_id AND cm.product_id = pr.product_id
    LEFT JOIN current_month_retailer_benchmark cmrb ON cm.warehouse_id = cmrb.warehouse_id AND cm.product_id = cmrb.product_id
    LEFT JOIN product_lookup pl ON pl.product_id = cm.product_id
    LEFT JOIN cat_brand_median cbm ON cbm.warehouse_id = cm.warehouse_id 
        AND cbm.cat = pl.cat 
        AND cbm.brand = pl.brand
    LEFT JOIN cat_median cm_median ON cm_median.warehouse_id = cm.warehouse_id 
        AND cm_median.cat = pl.cat
),

-- Pre-calculate combined ratio to avoid repetition
final_with_combined AS (
    SELECT
        crc.*,
        -- Calculate combined performance ratio once
        CASE WHEN (crc.yesterday_raw_weight + crc.recent_7d_raw_weight + crc.mtd_raw_weight) > 0
        THEN (
            (crc.yesterday_raw_weight / (crc.yesterday_raw_weight + crc.recent_7d_raw_weight + crc.mtd_raw_weight)) * crc.yesterday_ratio_capped +
            (crc.recent_7d_raw_weight / (crc.yesterday_raw_weight + crc.recent_7d_raw_weight + crc.mtd_raw_weight)) * crc.recent_ratio_capped +
            (crc.mtd_raw_weight / (crc.yesterday_raw_weight + crc.recent_7d_raw_weight + crc.mtd_raw_weight)) * crc.mtd_ratio_capped
        )
        ELSE 0 END AS combined_perf_ratio
    FROM combined_ratio_calc crc
)

-- Final output (same columns as original, values adjusted when over-achieving)
SELECT
    f.warehouse_id,
    f.product_id,
    
    -- Current period quantities
    f.yesterday_qty,
    f.yesterday_retailers,
    f.recent_7d_qty,
    f.recent_7d_in_stock_days,
    f.mtd_qty,
    f.mtd_in_stock_days,
    
    -- Quantity Benchmarks (P80) - adjusted when over-achieving, with fallback already applied
    CASE WHEN f.combined_perf_ratio > 1.1 
         THEN ROUND(f.p80_daily_240d + 0.5 * f.std_daily_240d, 2)
         ELSE f.p80_daily_240d 
    END AS p80_daily_240d,
    f.avg_daily_240d,
    f.std_daily_240d,
    f.in_stock_days_240d,
    CASE WHEN f.combined_perf_ratio > 1.1 
         THEN ROUND(f.p80_7d_sum_240d + 0.5 * f.std_daily_240d * 7, 2)
         ELSE f.p80_7d_sum_240d 
    END AS p80_7d_sum_240d,
    CASE WHEN f.combined_perf_ratio > 1.1 
         THEN ROUND(f.p80_mtd_12mo + 0.5 * f.std_daily_240d * (SELECT current_day_of_month FROM params), 2)
         ELSE f.p80_mtd_12mo 
    END AS p80_mtd_12mo,
    
    -- Retailer Benchmarks (P70) - adjusted when over-achieving
    CASE WHEN f.combined_perf_ratio > 1.1 
         THEN ROUND(f.p70_daily_retailers_240d + 0.5 * f.std_daily_retailers_240d, 2)
         ELSE f.p70_daily_retailers_240d 
    END AS p70_daily_retailers_240d,
    f.avg_daily_retailers_240d,
    f.std_daily_retailers_240d,
    
    -- Performance ratios (adjusted when over-achieving)
    ROUND(f.yesterday_qty / NULLIF(
        CASE WHEN f.combined_perf_ratio > 1.1 
             THEN f.p80_daily_240d + 0.5 * f.std_daily_240d
             ELSE f.p80_daily_240d 
        END, 0), 2) AS yesterday_ratio,
    ROUND(f.recent_7d_qty / NULLIF(
        CASE WHEN f.combined_perf_ratio > 1.1 
             THEN f.p80_7d_sum_240d + 0.5 * f.std_daily_240d * 7
             ELSE f.p80_7d_sum_240d 
        END, 0), 2) AS recent_ratio,
    ROUND(f.mtd_qty / NULLIF(
        CASE WHEN f.combined_perf_ratio > 1.1 
             THEN f.p80_mtd_12mo + 0.5 * f.std_daily_240d * (SELECT current_day_of_month FROM params)
             ELSE f.p80_mtd_12mo 
        END, 0), 2) AS mtd_ratio,
    ROUND(f.yesterday_retailers / NULLIF(
        CASE WHEN f.combined_perf_ratio > 1.1 
             THEN f.p70_daily_retailers_240d + 0.5 * f.std_daily_retailers_240d
             ELSE f.p70_daily_retailers_240d 
        END, 0), 2) AS yesterday_retailer_ratio,
    
    -- Additional columns for visibility
    ROUND(f.combined_perf_ratio, 2) AS combined_perf_ratio,
    CASE WHEN f.combined_perf_ratio > 1.1 THEN 1 ELSE 0 END AS is_over_achiever,
    f.is_new_sku  -- Flag to identify new SKUs using current month benchmarks

FROM final_with_combined f
WHERE f.warehouse_id IN (1, 236, 337, 8, 339, 170, 501, 401, 703, 632, 797, 962);


'''

# Execute benchmark query
print("Loading performance benchmark data (this may take a moment due to 240-day history)...")
df_benchmarks = query_snowflake(PERFORMANCE_BENCHMARK_QUERY)
df_benchmarks = convert_to_numeric(df_benchmarks)
print(f"Loaded {len(df_benchmarks)} benchmark records")

# =============================================================================
# Apply Minimum Thresholds to Benchmark Values
# - Daily quantity benchmarks should not be below 5
# - Daily retailers benchmarks should not be less than 2
# This ensures performance calculations don't use unrealistic benchmarks
# =============================================================================
MIN_DAILY_QTY_BENCHMARK = 5
MIN_DAILY_RETAILERS_BENCHMARK = 5

# Apply minimums to daily benchmarks
df_benchmarks['p80_daily_240d'] = df_benchmarks['p80_daily_240d'].clip(lower=MIN_DAILY_QTY_BENCHMARK)
df_benchmarks['p70_daily_retailers_240d'] = df_benchmarks['p70_daily_retailers_240d'].clip(lower=MIN_DAILY_RETAILERS_BENCHMARK)

# Apply proportional minimums to interval-based benchmarks
df_benchmarks['p80_7d_sum_240d'] = df_benchmarks['p80_7d_sum_240d'].clip(lower=MIN_DAILY_QTY_BENCHMARK * 7)  # 35

# For MTD, calculate dynamic minimum based on days in current period
# mtd_in_stock_days represents how many days of data we have in current month
df_benchmarks['p80_mtd_12mo'] = df_benchmarks.apply(
    lambda row: max(row['p80_mtd_12mo'], MIN_DAILY_QTY_BENCHMARK * max(row['mtd_in_stock_days'], 1)),
    axis=1
)

print(f"Applied minimum thresholds: qty >= {MIN_DAILY_QTY_BENCHMARK}/day, retailers >= {MIN_DAILY_RETAILERS_BENCHMARK}/day")

# Preview
df_benchmarks


Loading performance benchmark data (this may take a moment due to 240-day history)...


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Loaded 300661 benchmark records


Applied minimum thresholds: qty >= 5/day, retailers >= 5/day


,warehouse_id,product_id,yesterday_qty,yesterday_retailers,recent_7d_qty,recent_7d_in_stock_days,mtd_qty,mtd_in_stock_days,p80_daily_240d,avg_daily_240d,...,p70_daily_retailers_240d,avg_daily_retailers_240d,std_daily_retailers_240d,yesterday_ratio,recent_ratio,mtd_ratio,yesterday_retailer_ratio,combined_perf_ratio,is_over_achiever,is_new_sku
0,8,19555,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,0.00,0.00,NaN,0.00,0.00,0,0
1,1,3821,0,0,0,0,0,0,7.00,0.000000,...,5.00,0.000000,0.000000,0.00,0.00,NaN,0.00,0.00,0,0
2,797,18688,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,NaN,NaN,NaN,0.00,0.00,0,0
3,236,24324,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,NaN,NaN,NaN,0.00,0.00,0,0
4,8,13666,0,0,0,0,0,0,5.00,0.000000,...,5.00,0.000000,0.000000,0.00,0.00,NaN,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300656,962,201,337,56,1792,7,4582,12,352.22,193.950000,...,62.53,38.175000,24.468596,0.96,0.84,1.18,0.90,1.33,1,0
300657,962,11491,10,8,64,7,292,17,20.00,14.240741,...,12.00,10.000000,4.530817,0.50,0.66,1.92,0.67,1.01,0,0
300658,962,12772,0,0,0,0,0,0,9.40,5.333333,...,5.00,1.666667,2.081666,0.00,0.00,NaN,0.00,0.00,0,0
300659,962,12355,0,0,0,0,0,0,31.60,18.426471,...,6.00,4.838235,2.276488,0.00,0.00,0.00,0.00,0.00,0,0


In [40]:
# =============================================================================
# Add Performance Benchmarks and Tags to pricing_with_discount
# =============================================================================

# Merge benchmark data with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_benchmarks[[
        'warehouse_id', 'product_id',
        'yesterday_qty', 'yesterday_retailers', 'recent_7d_qty', 'recent_7d_in_stock_days', 'mtd_qty', 'mtd_in_stock_days',
        'p80_daily_240d', 'avg_daily_240d','std_daily_240d', 'in_stock_days_240d',
        'p80_7d_sum_240d', 'p80_mtd_12mo',
        'p70_daily_retailers_240d', 'avg_daily_retailers_240d', 'std_daily_retailers_240d',
        'yesterday_ratio', 'recent_ratio', 'mtd_ratio', 'yesterday_retailer_ratio'
    ]], 
    on=['warehouse_id', 'product_id'], 
    how='left'
)

# Fill NaN values
qty_cols = ['yesterday_qty', 'yesterday_retailers', 'recent_7d_qty', 'recent_7d_in_stock_days', 'mtd_qty', 'mtd_in_stock_days']
for col in qty_cols:
    pricing_with_discount[col] = pricing_with_discount[col].fillna(0)

benchmark_cols = ['p80_daily_240d', 'p80_7d_sum_240d', 'p80_mtd_12mo', 'p70_daily_retailers_240d']
for col in benchmark_cols:
    pricing_with_discount[col] = pricing_with_discount[col].fillna(5)  # Default to 1 to avoid division issues

ratio_cols = ['yesterday_ratio', 'recent_ratio', 'mtd_ratio', 'yesterday_retailer_ratio']
for col in ratio_cols:
    pricing_with_discount[col] = pricing_with_discount[col].fillna(0)

pricing_with_discount['avg_daily_240d'] = pricing_with_discount['avg_daily_240d'].fillna(0)
pricing_with_discount['in_stock_days_240d'] = pricing_with_discount['in_stock_days_240d'].fillna(0)
pricing_with_discount['avg_daily_retailers_240d'] = pricing_with_discount['avg_daily_retailers_240d'].fillna(0)
pricing_with_discount['std_daily_retailers_240d'] = pricing_with_discount['std_daily_retailers_240d'].fillna(0)

# =============================================================================
# Performance Tags - Classify each ratio
# =============================================================================
def get_performance_tag(ratio):
    """
    Classify performance based on ratio to benchmark
    On Track: 90% to 115% of benchmark
    Upper tiers: start from 115%
    Lower tiers: start from 90%
    """
    if pd.isna(ratio) or ratio == 0:
        return 'No Data'
    elif ratio >= 1.75:
        return 'Star Performer'      # 🌟 75%+ above benchmark
    elif ratio > 1.15:
        return 'Over Achiever'       # 🔥 15%+ above benchmark  
    elif ratio >= 0.90:
        return 'On Track'            # ✅ Meeting benchmark (90%-115%)
    elif ratio >= 0.70:
        return 'Underperforming'     # ⚠️ 10%-30% below benchmark
    elif ratio >= 0.40:
        return 'Struggling'          # 🔻 30%-60% below benchmark
    else:
        return 'Critical'            # 🚨 60%+ below benchmark

# Apply tags to each timeframe
pricing_with_discount['yesterday_status'] = pricing_with_discount['yesterday_ratio'].apply(get_performance_tag)
pricing_with_discount['recent_status'] = pricing_with_discount['recent_ratio'].apply(get_performance_tag)
pricing_with_discount['mtd_status'] = pricing_with_discount['mtd_ratio'].apply(get_performance_tag)

# =============================================================================
# Combined Performance Score (weighted average of ratios)
# Approach 2: Scale Weights by In-Stock Percentage
# =============================================================================

# Calculate days in month so far (excluding today)
days_in_month_so_far = max(TODAY.day - 1, 1)  # At least 1 to avoid division by zero

# Calculate in-stock percentages for each period
pricing_with_discount['yesterday_in_stock_pct'] = 1 - pricing_with_discount['oos_yesterday']
pricing_with_discount['recent_7d_in_stock_pct'] = pricing_with_discount['recent_7d_in_stock_days'] / 7
pricing_with_discount['mtd_in_stock_pct'] = pricing_with_discount['mtd_in_stock_days'] / days_in_month_so_far

# Base weights: Yesterday 20%, Recent 7d 40%, MTD 40%
# Scale by in-stock percentage
# NOTE: MTD weight = 0 for first 3 days of month (unreliable data)
MTD_RELIABLE_DAY = 3  # Only use MTD when day >= 3
pricing_with_discount['yesterday_raw_weight'] = 0.2 * pricing_with_discount['yesterday_in_stock_pct']
pricing_with_discount['recent_7d_raw_weight'] = 0.4 * pricing_with_discount['recent_7d_in_stock_pct']
pricing_with_discount['mtd_raw_weight'] = np.where(
    TODAY.day >= MTD_RELIABLE_DAY,
    0.4 * pricing_with_discount['mtd_in_stock_pct'],
    0  # Set MTD weight to 0 at start of month
)

# Total raw weight for normalization
pricing_with_discount['total_raw_weight'] = (
    pricing_with_discount['yesterday_raw_weight'] + 
    pricing_with_discount['recent_7d_raw_weight'] + 
    pricing_with_discount['mtd_raw_weight']
)

# Normalized weights (sum to 1)
pricing_with_discount['yesterday_norm_weight'] = np.where(
    pricing_with_discount['total_raw_weight'] > 0,
    pricing_with_discount['yesterday_raw_weight'] / pricing_with_discount['total_raw_weight'],
    0
)
pricing_with_discount['recent_7d_norm_weight'] = np.where(
    pricing_with_discount['total_raw_weight'] > 0,
    pricing_with_discount['recent_7d_raw_weight'] / pricing_with_discount['total_raw_weight'],
    0
)
pricing_with_discount['mtd_norm_weight'] = np.where(
    pricing_with_discount['total_raw_weight'] > 0,
    pricing_with_discount['mtd_raw_weight'] / pricing_with_discount['total_raw_weight'],
    0
)

# Combined performance ratio with dynamic weights based on in-stock days
pricing_with_discount['combined_perf_ratio'] = (
    pricing_with_discount['yesterday_norm_weight'] * pricing_with_discount['yesterday_ratio'].clip(upper=3) +
    pricing_with_discount['recent_7d_norm_weight'] * pricing_with_discount['recent_ratio'].clip(upper=3) +
    pricing_with_discount['mtd_norm_weight'] * pricing_with_discount['mtd_ratio'].clip(upper=3)
)

# Handle cases where total_raw_weight = 0 (completely OOS in all periods)
pricing_with_discount['combined_perf_ratio'] = pricing_with_discount['combined_perf_ratio'].fillna(0)

# Clean up intermediate columns (optional - keep for debugging)
weight_debug_cols = ['yesterday_in_stock_pct', 'recent_7d_in_stock_pct', 'mtd_in_stock_pct',
                     'yesterday_raw_weight', 'recent_7d_raw_weight', 'mtd_raw_weight', 'total_raw_weight',
                     'yesterday_norm_weight', 'recent_7d_norm_weight', 'mtd_norm_weight']
# Uncomment to drop: pricing_with_discount.drop(columns=weight_debug_cols, inplace=True)

print(f"\nDynamic weight calculation complete!")
print(f"Days in month so far: {days_in_month_so_far}")
print(f"\nSample of weight distributions:")
print(pricing_with_discount[pricing_with_discount['total_raw_weight'] > 0][
    ['product_id', 'warehouse_id', 'oos_yesterday', 'recent_7d_in_stock_days', 'mtd_in_stock_days',
     'yesterday_norm_weight', 'recent_7d_norm_weight', 'mtd_norm_weight', 'combined_perf_ratio']
].head(10))

pricing_with_discount['combined_status'] = pricing_with_discount['combined_perf_ratio'].apply(get_performance_tag)

# =============================================================================
# High Performer Flag (for immediate action consideration)
# =============================================================================
# Flag SKUs that are significantly over-achieving and may need action (price increase, etc.)
pricing_with_discount['high_performer_flag'] = np.where(
    (pricing_with_discount['yesterday_ratio'] >= 1.5) & 
    (pricing_with_discount['recent_ratio'] >= 1.3) &
    (pricing_with_discount['mtd_ratio'] >= 1.2),
    1, 0
)

# Star performer flag (exceptional - all metrics 2x+ benchmark)
pricing_with_discount['star_performer_flag'] = np.where(
    (pricing_with_discount['yesterday_ratio'] >= 2.0) & 
    (pricing_with_discount['recent_ratio'] >= 1.5) &
    (pricing_with_discount['mtd_ratio'] >= 1.5),
    1, 0
)

# =============================================================================
# Summary
# =============================================================================
print(f"Performance benchmarks added!")
print(f"\n{'='*60}")
print(f"PERFORMANCE STATUS DISTRIBUTION")
print(f"{'='*60}")

print(f"\nYesterday Status:")
print(pricing_with_discount['yesterday_status'].value_counts().to_string())

print(f"\nRecent 7d Status:")
print(pricing_with_discount['recent_status'].value_counts().to_string())

print(f"\nMTD Status:")
print(pricing_with_discount['mtd_status'].value_counts().to_string())

print(f"\nCombined Status:")
print(pricing_with_discount['combined_status'].value_counts().to_string())

print(f"\n{'='*60}")
print(f"HIGH PERFORMERS (Action Candidates)")
print(f"{'='*60}")
print(f"High Performers (flag=1): {len(pricing_with_discount[pricing_with_discount['high_performer_flag'] == 1])}")
print(f"Star Performers (flag=1): {len(pricing_with_discount[pricing_with_discount['star_performer_flag'] == 1])}")

# Show top performers
print(f"\nTop 15 Star Performers:")
pricing_with_discount[pricing_with_discount['star_performer_flag'] == 1].nlargest(15, 'combined_perf_ratio')[
    ['product_id', 'warehouse_id', 'sku', 
     'yesterday_ratio', 'recent_ratio', 'mtd_ratio', 'combined_perf_ratio',
     'yesterday_status', 'combined_status']
]



Dynamic weight calculation complete!
Days in month so far: 25

Sample of weight distributions:
   product_id  warehouse_id  oos_yesterday  recent_7d_in_stock_days  \
0        4940           797              1                        0   
1       22318           632              0                        1   
2         788           797              0                        4   
3       21683             1              0                        7   
4       21683             1              0                        7   
5       21683             1              0                        7   
6       21683             1              0                        7   
7       21683             1              0                        7   
8       21683             1              0                        7   
9        9786           236              0                        7   

   mtd_in_stock_days  yesterday_norm_weight  recent_7d_norm_weight  \
0                 10               0.000000         

,product_id,warehouse_id,sku,yesterday_ratio,recent_ratio,mtd_ratio,combined_perf_ratio,yesterday_status,combined_status
41213,8156,8,حفاضات بامبرز بانتس مقاس 4 - 58 حفاضة,3.22,4.49,3.00,3.000000,Star Performer,Star Performer
102036,9611,236,سمن روابى - 4.25 جم,20.75,7.49,3.92,3.000000,Star Performer,Star Performer
80760,23335,501,عصير تانج مانجو بودر برطمان - 450 جم,5.16,3.92,2.53,2.946213,Star Performer,Star Performer
45303,9396,8,لوكس صابون حلم السعاده / برتقالي - 165 جم,25.20,4.99,2.81,2.924000,Star Performer,Star Performer
95884,13118,8,أمريكانا فول سادة 20% خصم-( 12عرض*2عبوة ) 400 جم,7.34,3.63,2.50,2.897311,Star Performer,Star Performer
90711,13269,236,فاين مناديل تواليت كمفورت 24 رول20+ 4مجانا 24 رول,16.16,3.57,2.48,2.890526,Star Performer,Star Performer
3494,24567,703,مسحوق باهي يدوي لافندر - 60 جم,10.42,5.48,2.62,2.848000,Star Performer,Star Performer
90787,475,1,هانم سمنة بطعم الزبدة الصفراء- 1.5 كجم,6.21,2.66,2.87,2.812000,Star Performer,Star Performer
90788,475,1,هانم سمنة بطعم الزبدة الصفراء- 1.5 كجم,6.21,2.66,2.87,2.812000,Star Performer,Star Performer
34732,5097,962,ميزة دقيق - 1 كجم,3.25,2.79,2.29,2.793535,Star Performer,Star Performer


In [41]:
# =============================================================================
# No NMV in Last 4 Months Flag
# Identifies SKUs that have not generated any NMV in the past 4 months (120 days)
# =============================================================================
NO_NMV_4M_QUERY = f'''
WITH nmv_last_4m AS (
    SELECT 
        pso.warehouse_id,
        pso.product_id,
        SUM(pso.total_price) AS total_nmv_4m
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
        AND so.created_at::DATE < CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY pso.warehouse_id, pso.product_id
    HAVING SUM(pso.total_price) > 0
)
SELECT 
    warehouse_id,
    product_id,
    total_nmv_4m
FROM nmv_last_4m
'''

# Execute query
print("Loading SKUs with NMV in last 4 months...")
df_nmv_4m = query_snowflake(NO_NMV_4M_QUERY)
df_nmv_4m = convert_to_numeric(df_nmv_4m)
print(f"Found {len(df_nmv_4m)} SKU-warehouse combinations with NMV in last 4 months")

# Merge and create no_nmv_4m flag
pricing_with_discount = pricing_with_discount.merge(
    df_nmv_4m[['warehouse_id', 'product_id', 'total_nmv_4m']],
    on=['warehouse_id', 'product_id'],
    how='left'
)

# Flag SKUs with no NMV in last 4 months
# 1 = No NMV (should potentially be filtered), 0 = Has NMV
pricing_with_discount['no_nmv_4m'] = np.where(
    pricing_with_discount['total_nmv_4m'].isna() | (pricing_with_discount['total_nmv_4m'] == 0),
    1, 0
)

# Fill NaN for total_nmv_4m
pricing_with_discount['total_nmv_4m'] = pricing_with_discount['total_nmv_4m'].fillna(0)

print(f"\n{'='*60}")
print(f"NO NMV IN LAST 4 MONTHS ANALYSIS")
print(f"{'='*60}")
print(f"Total records: {len(pricing_with_discount)}")
print(f"SKUs with NO NMV in 4 months (no_nmv_4m=1): {len(pricing_with_discount[pricing_with_discount['no_nmv_4m'] == 1])}")
print(f"SKUs with NMV in 4 months (no_nmv_4m=0): {len(pricing_with_discount[pricing_with_discount['no_nmv_4m'] == 0])}")

# Show sample of SKUs with no NMV
print(f"\nSample SKUs with no NMV in last 4 months:")
pricing_with_discount[pricing_with_discount['no_nmv_4m'] == 1][
    ['product_id', 'warehouse_id', 'sku', 'stocks', 'in_stock_rr', 'zero_demand', 'no_nmv_4m']
].head(15)


Loading SKUs with NMV in last 4 months...


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Found 29400 SKU-warehouse combinations with NMV in last 4 months

NO NMV IN LAST 4 MONTHS ANALYSIS
Total records: 105318
SKUs with NO NMV in 4 months (no_nmv_4m=1): 69918
SKUs with NMV in 4 months (no_nmv_4m=0): 35400

Sample SKUs with no NMV in last 4 months:


,product_id,warehouse_id,sku,stocks,in_stock_rr,zero_demand,no_nmv_4m
11,10922,339,المطبخ مكرونة لسان عصفور- 400 جم,0,0.0,0,1
12,10922,170,المطبخ مكرونة لسان عصفور- 400 جم,0,0.0,0,1
17,22940,339,رو شيبس زبادي و خيار كرينكلز اكسترا - 15 جنية,0,0.0,0,1
18,22940,170,رو شيبس زبادي و خيار كرينكلز اكسترا - 15 جنية,0,0.0,0,1
19,79,401,زيت حلوة خليط - 700 مل,0,0.0,0,1
26,38,632,مكرونة حواء لسان عصفور - 400 جم,0,0.0,0,1
27,5020,797,تونة دولفين قطعة واحدة حار - 200 جم,0,0.0,0,1
34,9828,501,بن ابو عوف سادة وسط - 50 جم,0,0.0,0,1
35,25586,337,ايليت شوكولاتة بحشو كريمة البندق والكريسبي -...,0,0.0,0,1
36,25586,8,ايليت شوكولاتة بحشو كريمة البندق والكريسبي -...,0,0.0,0,1


In [42]:
# =============================================================================
# Normal Refill Query - Avg qty & stddev for frequent retailers (last 120 days)
# Frequent retailer definition based on ABC classification (from existing dataframe):
#   - Class A: bought 4+ times
#   - Class B: bought 3+ times
#   - Class C: bought 2+ times
# =============================================================================
NORMAL_REFILL_QUERY = f'''
WITH params AS (
    SELECT 
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE AS today,
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120 AS history_start
),

-- Get retailer order counts per product-warehouse (last 120 days)
retailer_orders AS (
    SELECT 
        pso.warehouse_id,
        pso.product_id,
        so.retailer_id,
        COUNT(DISTINCT so.id) AS order_count
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    CROSS JOIN params p
    WHERE so.created_at::DATE >= p.history_start
        AND so.created_at::DATE < p.today
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY pso.warehouse_id, pso.product_id, so.retailer_id
),

-- Get individual order quantities per retailer
order_quantities AS (
    SELECT 
        pso.warehouse_id,
        pso.product_id,
        so.retailer_id,
        so.id AS order_id,
        SUM(pso.purchased_item_count * pso.basic_unit_count) AS order_qty
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    CROSS JOIN params p
    WHERE so.created_at::DATE >= p.history_start
        AND so.created_at::DATE < p.today
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY pso.warehouse_id, pso.product_id, so.retailer_id, so.id
)

-- Return retailer-level data with order counts for Python filtering
SELECT 
    oq.warehouse_id,
    oq.product_id,
    oq.retailer_id,
    ro.order_count,
    oq.order_id,
    oq.order_qty
FROM order_quantities oq
JOIN retailer_orders ro 
    ON ro.warehouse_id = oq.warehouse_id 
    AND ro.product_id = oq.product_id 
    AND ro.retailer_id = oq.retailer_id
'''

# Execute normal refill query
print("Loading retailer order data for normal refill calculation (last 120 days)...")
df_retailer_orders = query_snowflake(NORMAL_REFILL_QUERY)
df_retailer_orders = convert_to_numeric(df_retailer_orders)
print(f"Loaded {len(df_retailer_orders)} retailer order records")

# Get ABC classification from existing dataframe
abc_mapping = pricing_with_discount[['warehouse_id', 'product_id', 'abc_class']].drop_duplicates()
print(f"ABC classification mapping: {len(abc_mapping)} product-warehouse combinations")

# Merge ABC classification into retailer orders
df_retailer_orders = df_retailer_orders.merge(
    abc_mapping,
    on=['warehouse_id', 'product_id'],
    how='inner'
)
print(f"Records after ABC merge: {len(df_retailer_orders)}")

# Filter frequent retailers based on ABC class thresholds
# Class A: 4+ orders, Class B: 3+ orders, Class C: 2+ orders
df_frequent = df_retailer_orders[
    ((df_retailer_orders['abc_class'] == 'A') & (df_retailer_orders['order_count'] >= 4)) |
    ((df_retailer_orders['abc_class'] == 'B') & (df_retailer_orders['order_count'] >= 3)) |
    ((df_retailer_orders['abc_class'] == 'C') & (df_retailer_orders['order_count'] >= 2))
].copy()
print(f"Records from frequent retailers: {len(df_frequent)}")

# Calculate normal_refill (avg qty) and refill_stddev per product-warehouse
df_normal_refill = df_frequent.groupby(['warehouse_id', 'product_id']).agg(
    frequent_retailer_count=('retailer_id', 'nunique'),
    frequent_order_count=('order_id', 'nunique'),
    normal_refill=('order_qty', 'mean'),
    refill_stddev=('order_qty', 'std')
).reset_index()

# Round values and fill NaN stddev (when only 1 order)
df_normal_refill['normal_refill'] = df_normal_refill['normal_refill'].round(2)
df_normal_refill['refill_stddev'] = df_normal_refill['refill_stddev'].fillna(0).round(2)

# Filter to products with at least 2 orders for meaningful stats
df_normal_refill = df_normal_refill[df_normal_refill['frequent_order_count'] >= 2]
print(f"Final normal refill records (min 2 orders): {len(df_normal_refill)}")

# Merge with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_normal_refill[['warehouse_id', 'product_id', 'frequent_retailer_count', 
                      'frequent_order_count', 'normal_refill', 'refill_stddev']],
    on=['warehouse_id', 'product_id'],
    how='left'
)

# Fill NaN values
pricing_with_discount['frequent_retailer_count'] = pricing_with_discount['frequent_retailer_count'].fillna(0)
pricing_with_discount['frequent_order_count'] = pricing_with_discount['frequent_order_count'].fillna(0)
pricing_with_discount['normal_refill'] = pricing_with_discount['normal_refill'].fillna(0)
pricing_with_discount['refill_stddev'] = pricing_with_discount['refill_stddev'].fillna(0)

print(f"\n{'='*60}")
print(f"NORMAL REFILL ANALYSIS (Frequent Retailers - 120 days)")
print(f"{'='*60}")
print(f"Records with normal_refill data: {len(pricing_with_discount[pricing_with_discount['normal_refill'] > 0])}")
print(f"Records without normal_refill data: {len(pricing_with_discount[pricing_with_discount['normal_refill'] == 0])}")
print(f"\nNormal refill distribution:")
print(pricing_with_discount[pricing_with_discount['normal_refill'] > 0]['normal_refill'].describe())
print(f"\nSample data:")
pricing_with_discount[pricing_with_discount['normal_refill'] > 0][
    ['product_id', 'warehouse_id', 'sku', 'abc_class', 'frequent_retailer_count', 
     'frequent_order_count', 'normal_refill', 'refill_stddev', 'in_stock_rr']
].head(15)


Loading retailer order data for normal refill calculation (last 120 days)...


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Loaded 4907931 retailer order records
ABC classification mapping: 87123 product-warehouse combinations


Records after ABC merge: 4675394


Records from frequent retailers: 1615430


Final normal refill records (min 2 orders): 22588

NORMAL REFILL ANALYSIS (Frequent Retailers - 120 days)
Records with normal_refill data: 28393
Records without normal_refill data: 76925

Normal refill distribution:


count    28393.000000
mean         3.401020
std         29.836308
min          1.000000
25%          1.300000
50%          1.910000
75%          3.270000
max       4534.000000
Name: normal_refill, dtype: float64

Sample data:


,product_id,warehouse_id,sku,abc_class,frequent_retailer_count,frequent_order_count,normal_refill,refill_stddev,in_stock_rr
0,4940,797,مناديل فاميليا مطبخ كلاسيك 6 رول + 2 رول هدية ...,B,25.0,104.0,2.37,2.26,5.0
1,22318,632,اوريو بسكوت بكريمة بطعم الشوكولاتة والبندق - ...,C,3.0,8.0,1.25,0.46,1.0
3,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,B,37.0,153.0,3.14,1.82,17.0
4,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,B,37.0,153.0,3.14,1.82,17.0
5,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,B,37.0,153.0,3.14,1.82,17.0
6,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,B,37.0,153.0,3.14,1.82,17.0
7,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,B,37.0,153.0,3.14,1.82,17.0
8,21683,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,B,37.0,153.0,3.14,1.82,17.0
9,9786,236,عصير جهينة بيور جوافة كوكتيل - 235 مل,C,51.0,134.0,1.16,0.49,2.0
10,9786,962,عصير جهينة بيور جوافة كوكتيل - 235 مل,B,27.0,108.0,1.05,0.21,4.0


In [43]:
# =============================================================================
# Live Cart Rules Query - Get current cart rules from the system
# Merges on product_id and cohort_id
# =============================================================================
LIVE_CART_RULES_QUERY = f'''
SELECT 
    cppu.cohort_id,
    pup.product_id,
    pup.packing_unit_id,
    pup.basic_unit_count,
    COALESCE(cppu.MAX_PER_SALES_ORDER, cppu2.MAX_PER_SALES_ORDER) AS current_cart_rule
FROM COHORT_PRODUCT_PACKING_UNITS cppu 
JOIN PACKING_UNIT_PRODUCTS pup ON cppu.PRODUCT_PACKING_UNIT_ID = pup.id 
JOIN cohorts c ON c.id = cppu.cohort_id
LEFT JOIN COHORT_PRODUCT_PACKING_UNITS cppu2 
    ON cppu.PRODUCT_PACKING_UNIT_ID = cppu2.PRODUCT_PACKING_UNIT_ID 
    AND cppu2.cohort_id = c.FALLBACK_COHORT_ID
WHERE cppu.cohort_id IN ({','.join(map(str, COHORT_IDS))})
'''

# Execute live cart rules query
print("Loading live cart rules...")
df_cart_rules = query_snowflake(LIVE_CART_RULES_QUERY)
df_cart_rules = convert_to_numeric(df_cart_rules)
print(f"Loaded {len(df_cart_rules)} cart rule records")

# Aggregate to product-cohort level (take the cart rule for basic unit, or min if multiple)
# Filter to basic unit (packing_unit_id where basic_unit_count = 1) for simpler merging
df_cart_rules_basic = df_cart_rules[df_cart_rules['basic_unit_count'] == 1].copy()
print(f"Basic unit cart rules: {len(df_cart_rules_basic)}")

# If no basic unit, take the minimum cart rule per product-cohort
df_cart_rules_agg = df_cart_rules.groupby(['cohort_id', 'product_id']).agg(
    current_cart_rule=('current_cart_rule', 'min')
).reset_index()

# Prefer basic unit cart rule, fallback to aggregated
df_cart_rules_final = df_cart_rules_basic[['cohort_id', 'product_id', 'current_cart_rule']].drop_duplicates()
df_cart_rules_final = df_cart_rules_final.merge(
    df_cart_rules_agg[['cohort_id', 'product_id', 'current_cart_rule']].rename(columns={'current_cart_rule': 'cart_rule_agg'}),
    on=['cohort_id', 'product_id'],
    how='outer'
)
df_cart_rules_final['current_cart_rule'] = df_cart_rules_final['current_cart_rule'].fillna(df_cart_rules_final['cart_rule_agg'])
df_cart_rules_final = df_cart_rules_final[['cohort_id', 'product_id', 'current_cart_rule']].drop_duplicates()
print(f"Final cart rules (product-cohort level): {len(df_cart_rules_final)}")

# Merge with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_cart_rules_final,
    on=['cohort_id', 'product_id'],
    how='left'
)

# Fill NaN cart rules with 0 (no cart rule set)
pricing_with_discount['current_cart_rule'] = pricing_with_discount['current_cart_rule'].fillna(0)

print(f"\n{'='*60}")
print(f"LIVE CART RULES ANALYSIS")
print(f"{'='*60}")
print(f"Records with cart rule > 0: {len(pricing_with_discount[pricing_with_discount['current_cart_rule'] > 0])}")
print(f"Records without cart rule: {len(pricing_with_discount[pricing_with_discount['current_cart_rule'] == 0])}")
print(f"\nCart rule distribution:")
print(pricing_with_discount[pricing_with_discount['current_cart_rule'] > 0]['current_cart_rule'].describe())
print(f"\nSample data with cart rules:")
pricing_with_discount[pricing_with_discount['current_cart_rule'] > 0][
    ['product_id', 'cohort_id', 'warehouse_id', 'sku', 'current_price', 'current_cart_rule', 'in_stock_rr']
].head(15)


Loading live cart rules...


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Loaded 111336 cart rule records
Basic unit cart rules: 73438
Final cart rules (product-cohort level): 73429



LIVE CART RULES ANALYSIS
Records with cart rule > 0: 105358
Records without cart rule: 0

Cart rule distribution:
count    105358.000000
mean         52.538924
std         565.221001
min           2.000000
25%          10.000000
50%          15.000000
75%          25.000000
max       10000.000000
Name: current_cart_rule, dtype: float64

Sample data with cart rules:


,product_id,cohort_id,warehouse_id,sku,current_price,current_cart_rule,in_stock_rr
0,4940,702,797,مناديل فاميليا مطبخ كلاسيك 6 رول + 2 رول هدية ...,252.75,10,5.0
1,22318,1125,632,اوريو بسكوت بكريمة بطعم الشوكولاتة والبندق - ...,90.25,5,1.0
2,788,702,797,حلوانى مربى فراولة- 340 جم,383.00,25,1.0
3,21683,700,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,46.00,30,17.0
4,21683,700,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,46.00,30,17.0
5,21683,700,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,46.00,30,17.0
6,21683,700,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,46.00,30,17.0
7,21683,700,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,46.00,30,17.0
8,21683,700,1,لمبادا ويفر محشو كريمه الشكولاته 5 اكس - 5 جنية,46.00,30,17.0
9,9786,701,236,عصير جهينة بيور جوافة كوكتيل - 235 مل,406.00,16,2.0


In [44]:
# =============================================================================
# Commercial Constraint Minimum Price Query
# Gets the minimum price constraints from finance.minimum_prices
# =============================================================================
COMMERCIAL_MIN_PRICE_QUERY = f'''
WITH to_remove AS (
    SELECT 
        check_date AS start_date,
        (check_date + INTERVAL '1 month') + 6 AS end_date 
    FROM (
        SELECT 
            CASE 
                WHEN DATE_PART('day', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) < 7 
                THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
                ELSE DATE_FROM_PARTS(
                    YEAR(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE), 
                    MONTH(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE), 
                    1
                )  
            END AS check_date
    )
),
region_mapping as(
select *
from (
values
('Greater Cairo', 'Cairo'),
('Greater Cairo', 'Giza')

)x(region,main_region)

)

SELECT  
    sku_id AS product_id,
    sku,
    brand AS comm_brand,
    cat AS comm_cat,
    region,
    created_at AS comm_created_at,
    min_price AS commercial_min_price
FROM (
    SELECT 
        product_id AS sku_id,
        product_name AS sku,
        brand,
        category AS cat,
        coalesce(rm.main_region,mp.region) as region,
        min_price,
        created_at,
        MAX(created_at) OVER (PARTITION BY product_id, mp.region) AS latest_date
    FROM finance.minimum_prices mp
	left join region_mapping rm on mp.region = rm.region
    WHERE is_deleted = 'false'
        AND created_at BETWEEN (SELECT start_date FROM to_remove) AND (SELECT end_date FROM to_remove)
) comm
WHERE created_at = latest_date
'''

# Execute commercial min price query
print("Loading commercial minimum price constraints...")
df_commercial_min = query_snowflake(COMMERCIAL_MIN_PRICE_QUERY)
df_commercial_min = convert_to_numeric(df_commercial_min)
print(f"Loaded {len(df_commercial_min)} commercial min price records")

# Merge with pricing_with_discount on product_id and region
pricing_with_discount = pricing_with_discount.merge(
    df_commercial_min[['product_id', 'region', 'commercial_min_price']],
    on=['product_id', 'region'],
    how='left'
)

# Fill NaN with 0 (no commercial constraint)
pricing_with_discount['commercial_min_price'] = pricing_with_discount['commercial_min_price'].fillna(0)

print(f"\n{'='*60}")
print(f"COMMERCIAL MINIMUM PRICE CONSTRAINTS")
print(f"{'='*60}")
print(f"Records with commercial min price: {len(pricing_with_discount[pricing_with_discount['commercial_min_price'] > 0])}")
print(f"Records without commercial min price: {len(pricing_with_discount[pricing_with_discount['commercial_min_price'] == 0])}")
print(f"\nCommercial min price distribution:")
print(pricing_with_discount[pricing_with_discount['commercial_min_price'] > 0]['commercial_min_price'].describe())
print(f"\nSample data with commercial constraints:")
pricing_with_discount[pricing_with_discount['commercial_min_price'] > 0][
    ['product_id', 'region', 'warehouse_id', 'sku', 'current_price', 'commercial_min_price', 'price_after_discount']
].head(15)


Loading commercial minimum price constraints...


Loaded 1994 commercial min price records

COMMERCIAL MINIMUM PRICE CONSTRAINTS
Records with commercial min price: 5277
Records without commercial min price: 100081

Commercial min price distribution:
count    5277.000000
mean      254.677016
std       196.717227
min        11.750000
25%       101.000000
50%       210.999999
75%       379.000000
max      1300.000000
Name: commercial_min_price, dtype: float64

Sample data with commercial constraints:


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,product_id,region,warehouse_id,sku,current_price,commercial_min_price,price_after_discount
15,3889,Alexandria,797,عصير سن توب توت مشكل - 250 مل,255.00,255.000000,253.145189
16,3889,Alexandria,797,عصير سن توب توت مشكل - 250 مل,255.00,255.000000,253.145189
66,13965,Upper Egypt,401,بيتي عصير جوافة عرض خاص27ج - 1 لتر,286.00,286.000000,286.000000
91,13964,Upper Egypt,703,بيتي عصير كوكتيل عرض خاص 27ج - 1 لتر,302.25,286.000000,301.549139
97,143,Delta East,339,عصير بيتى تفاح - 235 مل,189.75,186.000000,188.895262
98,143,Delta East,170,عصير بيتى تفاح - 235 مل,189.75,186.000000,187.970125
103,2419,Upper Egypt,401,حفاضات بى بم بانتس جامبو مقاس 4 - 56 حفاضة,274.25,274.348604,262.374967
104,2419,Alexandria,797,حفاضات بى بم بانتس جامبو مقاس 4 - 56 حفاضة,272.00,268.861632,260.494400
141,4043,Upper Egypt,401,حفاضات بى بم جامبو بانتس مقاس 6 - 46 حفاضة,272.25,272.139054,261.006100
142,4043,Cairo,1,حفاضات بى بم جامبو بانتس مقاس 6 - 46 حفاضة,266.75,266.696273,265.268044


In [45]:
# =============================================================================
# Active SKU Discount Query - Get current SKU discount percentage per warehouse
# =============================================================================
ACTIVE_SKU_DISCOUNT_QUERY = f'''
WITH active_sku_discount AS ( 
    SELECT 
        x.id AS sku_discount_id,
        retailer_id,
        product_id,
        packing_unit_id,
        DISCOUNT_PERCENTAGE,
        start_at,
        end_at 
    FROM (
        SELECT 
            sd.*,
            f.value::INT AS retailer_id 
        FROM SKU_DISCOUNTS sd,
        LATERAL FLATTEN(
            input => SPLIT(
                REPLACE(REPLACE(REPLACE(sd.retailer_ids, '{{', ''), '}}', ''), '"', ''), 
                ','
            )
        ) f
        WHERE start_at::DATE <= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
            AND end_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
    ) x 
    JOIN SKU_DISCOUNT_VALUES sdv ON x.id = sdv.sku_discount_id
    WHERE name_en = 'Special Discounts'
    QUALIFY MAX(start_at) OVER (PARTITION BY retailer_id, product_id, packing_unit_id) = start_at 
)

SELECT 
    product_id, 
    warehouse_id,
    AVG(DISCOUNT_PERCENTAGE) AS active_sku_disc_pct 
FROM (
    SELECT 
        asd.*,
        warehouse_id 
    FROM active_sku_discount asd 
    JOIN materialized_views.retailer_polygon rp ON rp.retailer_id = asd.retailer_id
    JOIN WAREHOUSE_DISPATCHING_RULES wdr ON wdr.product_id = asd.product_id
    JOIN DISPATCHING_POLYGONS dp ON dp.id = wdr.DISPATCHING_POLYGON_ID AND dp.district_id = rp.district_id
)
GROUP BY ALL
'''

# Execute active SKU discount query
print("Loading active SKU discount data...")
df_active_sku_disc = query_snowflake(ACTIVE_SKU_DISCOUNT_QUERY)
df_active_sku_disc = convert_to_numeric(df_active_sku_disc)
print(f"Loaded {len(df_active_sku_disc)} active SKU discount records")

# Merge with pricing_with_discount
pricing_with_discount = pricing_with_discount.merge(
    df_active_sku_disc[['product_id', 'warehouse_id', 'active_sku_disc_pct']],
    on=['product_id', 'warehouse_id'],
    how='left'
)

# Fill NaN with 0 (no active SKU discount)
pricing_with_discount['active_sku_disc_pct'] = pricing_with_discount['active_sku_disc_pct'].fillna(0)

print(f"\n{'='*60}")
print(f"ACTIVE SKU DISCOUNT ANALYSIS")
print(f"{'='*60}")
print(f"Records with active SKU discount: {len(pricing_with_discount[pricing_with_discount['active_sku_disc_pct'] > 0])}")
print(f"Records without active SKU discount: {len(pricing_with_discount[pricing_with_discount['active_sku_disc_pct'] == 0])}")
print(f"\nActive SKU discount distribution:")
print(pricing_with_discount[pricing_with_discount['active_sku_disc_pct'] > 0]['active_sku_disc_pct'].describe())
print(f"\nSample data with active SKU discounts:")
pricing_with_discount[pricing_with_discount['active_sku_disc_pct'] > 0][
    ['product_id', 'warehouse_id', 'sku', 'current_price', 'active_sku_disc_pct', 'discount_perc']
].head(15)


Loading active SKU discount data...


/tmp/ipykernel_25840/1273908998.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Loaded 10088 active SKU discount records

ACTIVE SKU DISCOUNT ANALYSIS
Records with active SKU discount: 12979
Records without active SKU discount: 92379

Active SKU discount distribution:
count    12979.000000
mean         1.561865
std          1.163708
min          0.250000
25%          0.730145
50%          1.061898
75%          2.084656
max          5.000000
Name: active_sku_disc_pct, dtype: float64

Sample data with active SKU discounts:


,product_id,warehouse_id,sku,current_price,active_sku_disc_pct,discount_perc
9,9786,236,عصير جهينة بيور جوافة كوكتيل - 235 مل,406.00,1.040175,0.003100
10,9786,962,عصير جهينة بيور جوافة كوكتيل - 235 مل,406.00,1.036417,0.003055
13,8635,703,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,85.00,0.330000,0.011800
14,362,1,عصير جهينة مانجو - 235 مل,209.50,0.319314,0.000484
15,3889,797,عصير سن توب توت مشكل - 250 مل,255.00,1.687064,0.007274
16,3889,797,عصير سن توب توت مشكل - 250 مل,255.00,1.687064,0.007274
28,8635,236,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,86.25,0.840124,0.014605
29,8635,236,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,86.25,0.840124,0.014605
30,8635,962,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,86.25,0.842021,0.001019
31,8635,962,لبان ترايدنت توت و فراولة 65 قطعة- 1.5 جنية,86.25,0.842021,0.001019


In [46]:
final_df = pricing_with_discount[(pricing_with_discount['no_nmv_4m']==0)|(pricing_with_discount['stocks']>0)]

In [47]:
# Drop duplicates before saving
final_df = final_df.drop_duplicates(subset=['product_id', 'warehouse_id'], keep='first')
final_df.to_excel('pricing_with_discount.xlsx', index=False)
print(f"Exported {len(final_df)} records (duplicates removed)")

Exported 28850 records (duplicates removed)


In [48]:
final_df['created_at'] = TODAY
final_df['created_at'] =pd.to_datetime(final_df['created_at']).dt.date

In [49]:
!pip install slack_sdk

In [50]:
status = upload_dataframe_to_snowflake("Egypt", final_df, "MATERIALIZED_VIEWS", "Pricing_data_extraction", "append", auto_create_table=True, conn=None)

# Send Slack notification
if status:
    slack_message = f"""✅ *Data Extraction Script Completed Successfully*
    
📅 Date: {TODAY}
📊 Records uploaded: {len(final_df):,}
🗄️ Table: MATERIALIZED_VIEWS.Pricing_data_extraction
⏰ Completed at: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')} Cairo time"""
    
    send_text_slack('new-pricing-logic',slack_message)
    print("✅ Slack notification sent!")
else:
    error_message = f"""❌ *Data Extraction Script Failed*
    
📅 Date: {TODAY}
⏰ Failed at: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')} Cairo time
⚠️ Upload to Snowflake failed - please check logs"""
    
    send_text_slack('new-pricing-logic',error_message)
    print("❌ Error notification sent to Slack!")

/home/ec2-user/service_account_key.json


/home/ec2-user/SageMaker/Pricing Runs/Prediction_Scripts_2/Happy_hour/git/Mustafa/Pricing Logic/common_functions.py:760: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, _, _, _ = write_pandas(


/home/ec2-user/service_account_key.json


Message Sent
✅ Slack notification sent!
